# Explainable and Trustworthy Bridge Damage Segmentation on dacl10k

**Foundation-model backbones, faithfulness-tested explanations and calibrated uncertainty for multi-label reinforced-concrete defect segmentation**

Companion code for the manuscript *Beyond accuracy: calibrated uncertainty, faithful explanations and robustness for multi-label bridge damage segmentation on dacl10k* (M. Ali, U. Nasir).

This notebook runs the full experimental programme of the study, from raw data to print-ready figures and tables. Every section opens with a short explanation of *what* is done and *why*.

| Research question | What this notebook does | Section |
|---|---|---|
| **RQ1** Do foundation backbones beat CNN baselines? | Trains FPN (EfficientNet-B4), SegFormer-B2, DINOv2-B + LoRA and DINOv2-B with full fine-tuning under identical conditions; paired bootstrap tests | 8–9 |
| **RQ2** Are explanations faithful? | Seg-Grad-CAM vs. occlusion; deletion and insertion tests, off-target leakage, model-randomisation test; class-confusion analysis | 12 |
| **RQ3** Which uncertainty estimate is best calibrated? | Single model, MC dropout, test-time augmentation, deep ensemble; per-class temperature scaling | 10 |
| **RQ4** Does uncertainty locate errors? | Risk–coverage analysis and error recall at fixed review budgets | 11 |
| Robustness | Blur, noise, low light and JPEG compression at three severities | 13 |

**Data licence.** dacl10k is released under CC BY-NC 4.0 (non-commercial use only). Cite: J. Flotzinger, P.J. Rösch, T. Braml, dacl10k: Benchmark for semantic bridge damage segmentation, in: Proceedings of the IEEE/CVF Winter Conference on Applications of Computer Vision (WACV), 2024.

---
## ▶ How to run

1. Open the notebook in Google Colab, in **one browser tab only**.
2. Choose **Runtime → Change runtime type → GPU**. The paper's final run used an NVIDIA A100 (40 GB).
3. Optional but recommended: set `SMOKE_TEST = True` in CELL 04 and choose **Runtime → Run all**. This dry run uses a few synthetic images and checks that every cell works in a few minutes.
4. Set `SMOKE_TEST = False` and choose **Runtime → Run all** for the real experiment.

Everything that must survive a session (splits, checkpoints, results, tables, figures) is saved in Google Drive under `MyDrive/dacl10k_project`. Training is resumable: if Colab disconnects, run all cells again and finished runs are skipped. A run from scratch trains 12 models (4 models × 3 seeds) and needs many GPU hours.

| Switch | Value used for the paper | Cell | Meaning |
|---|---|---|---|
| `SMOKE_TEST` | `False` | CELL 04 | `True` = few-minute dry run on synthetic data |
| `E1_SEEDS` | `[0, 1, 2]` | CELL 21 | Seeds trained for each of the four models |
| `RUN_EPOCH_CHECK` | `True` | CELL 26b | Training-length check (60 vs 30 epochs, full fine-tuning, weakest seed) |
| `RUN_E2` | off | CELL 26 | Optional higher-resolution study; not reported in the paper |
| `RELEASE_GPU_AT_END` | `True` | CELL 38 | Disconnects the runtime when all cells have finished |

**Numbering of outputs.** Tables 1–9 and Figures S1–S3 correspond to the same numbers in the paper. The notebook numbers the supplementary tables S1, S2, S4 and S5 (S3 is reserved for the optional resolution study); in the paper these are Tables S1, S2, S3 and S4. The notebook's Figures 2, 3 and 4–9 are the paper's Figures 2, 3 and 5–10; the notebook's Figure 1 (overview) is not used in the paper, whose Figures 1 (workflow) and 4 (model architecture) are in the repository's `docs/` folder.


---
## 1. Environment setup

### 1.1 Packages

Colab already ships PyTorch, OpenCV, scikit-learn, pandas and matplotlib. We add **segmentation-models-pytorch** (FPN baseline), **transformers** (SegFormer, DINOv2), **peft** (LoRA), **albumentations** (augmentation), **imagehash** (near-duplicate audit) and **python-docx** (Word report).

The cell also **removes `torchao`**: Colab preinstalls an old version that makes `peft` refuse to add LoRA adapters. Nothing in this notebook uses it.

In [ ]:
# CELL 01 · Install packages
import os, sys, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install segmentation-models-pytorch timm transformers peft albumentations imagehash python-docx
    # Colab preinstalls an old torchao that makes peft (LoRA) refuse to run; this notebook does not use it.
    !pip -q uninstall -y torchao

try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout or "No NVIDIA GPU visible.")
except FileNotFoundError:
    print("nvidia-smi not found: running on CPU.")
print("Running in Colab:", IN_COLAB)

### 1.2 Persistent storage

The Colab machine is wiped when a session ends. The dataset and its cache live on the local disk (fast, re-created each session); everything you cannot afford to lose (splits, checkpoints, results, figures, tables) goes to **Google Drive**.

In [ ]:
# CELL 02 · Mount Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_DATA_ROOT = "/content/dacl10k"
    DEFAULT_PROJECT_DIR = "/content/drive/MyDrive/dacl10k_project"
else:
    DEFAULT_DATA_ROOT = "./dacl10k_data"
    DEFAULT_PROJECT_DIR = "./dacl10k_project"

DATA_ROOT = os.environ.get("DACL_DATA_ROOT", DEFAULT_DATA_ROOT)
PROJECT_DIR = os.environ.get("DACL_PROJECT_DIR", DEFAULT_PROJECT_DIR)
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Data root   :", os.path.abspath(DATA_ROOT))
print("Project dir :", os.path.abspath(PROJECT_DIR))

### 1.3 Imports

In [ ]:
# CELL 03 · Imports
import json, glob, math, time, copy, random, hashlib, shutil, zipfile, warnings, re
from dataclasses import dataclass, field, asdict
from multiprocessing import get_context

import numpy as np
import pandas as pd
import cv2
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.lines import Line2D
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, Markdown

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
import segmentation_models_pytorch as smp
import imagehash
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore", category=UserWarning)
cv2.setNumThreads(0)  # avoid thread oversubscription inside DataLoader workers

import transformers, peft
print(f"torch {torch.__version__} | smp {smp.__version__} | transformers {transformers.__version__} "
      f"| peft {peft.__version__} | albumentations {A.__version__} | matplotlib {mpl.__version__}")

### 1.4 Configuration

**The first line of the next cell is the dry-run switch:** `SMOKE_TEST = True` runs everything in a few minutes on fake data; `SMOKE_TEST = False` is the real experiment.

| Setting | Value | Reason |
|---|---|---|
| `img_size` | 512 | Matches the published dacl10k baselines |
| `epochs` | 30 | Matches the baseline protocol; early stopping guards against overfitting |
| classes | 19, toolkit order | Same channel order as the official `dacl10k-toolkit` |
| `val_fraction` | 0.10 | Internal validation split drawn from the official training set (Section 5) |
| `seeds` | 0, 1, 2 | Three seeds per model for the final paper |
| `lora_r`, `lora_alpha` | 16, 32 | Common LoRA defaults |

In [ ]:
# CELL 04 · Configuration (SMOKE_TEST switch is the first line)
SMOKE_TEST = False   # <<< True = 5-minute dry run on fake data; False = real experiment
if os.environ.get("DACL_SMOKE") == "1":   # used only for automated testing of this notebook
    SMOKE_TEST = True
TRAIN_ONLY = False   # True = train only, without writing tables, figures or the report

CLASSES = ['Crack', 'ACrack', 'Wetspot', 'Efflorescence', 'Rust', 'Rockpocket', 'Hollowareas', 'Cavity',
           'Spalling', 'Graffiti', 'Weathering', 'Restformwork', 'ExposedRebars',
           'Bearing', 'EJoint', 'Drainage', 'PEquipment', 'JTape', 'WConccor']
C = len(CLASSES)
CLASS_INDEX = {c: i for i, c in enumerate(CLASSES)}

# Confusable groups singled out by the dacl10k authors (used in Section 12)
CONFUSABLE_PAIRS = [("Spalling", "Rockpocket"), ("Spalling", "WConccor"), ("Rockpocket", "WConccor"),
                    ("Weathering", "Wetspot"), ("Crack", "ACrack")]

@dataclass
class Config:
    data_root: str = DATA_ROOT
    project_dir: str = PROJECT_DIR
    img_size: int = 512
    batch_size: int = 8
    num_workers: int = min(8, os.cpu_count() or 2)   # parallel image loaders; speed only, no effect on the method
    epochs: int = 30
    patience: int = 8                 # early-stopping patience (epochs without val mIoU gain)
    lr_head: float = 3e-4
    lr_backbone: float = 5e-5         # pretrained encoders get a smaller learning rate
    weight_decay: float = 1e-4
    warmup_epochs: int = 2
    accum_steps: int = 1              # gradient accumulation (raise if you lower batch_size)
    amp: bool = True
    grad_clip: float = 1.0
    focal_weight: float = 10.0        # weight of focal BCE relative to Dice
    aux_weight: float = 0.1           # weight of the image-level auxiliary loss (as in the baseline)
    seeds: list = field(default_factory=lambda: [0, 1, 2])
    val_fraction: float = 0.10
    pretrained: bool = True
    fpn_encoder: str = "efficientnet-b4"
    segformer_name: str = "nvidia/mit-b2"
    dino_name: str = "facebook/dinov2-base"
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    grad_checkpointing: bool = False  # set True for DINOv2 on a 16 GB GPU
    # evaluation / uncertainty
    thr_grid: list = field(default_factory=lambda: [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)])
    pixels_per_image: int = 1024      # pixels sampled per image for calibration analysis
    mc_samples: int = 20
    tta_scales: list = field(default_factory=lambda: [0.75, 1.0, 1.25])
    # explanations / robustness
    n_xai_images: int = 40
    occlusion_patch: int = 32
    n_robust_images: int = 300
    phash_max_dist: int = 8           # Hamming distance treated as near-duplicate

cfg = Config()

if SMOKE_TEST:
    cfg.data_root = os.path.join(DATA_ROOT, "_synthetic")
    cfg.project_dir = os.path.join(PROJECT_DIR, "_smoke")
    cfg.img_size, cfg.batch_size, cfg.num_workers = 128, 4, 0
    cfg.epochs = int(os.environ.get("DACL_SMOKE_EPOCHS", "2"))
    cfg.patience, cfg.warmup_epochs = cfg.epochs, 1
    cfg.lr_backbone = cfg.lr_head      # nothing is pretrained in the smoke test
    cfg.seeds = [0, 1]
    cfg.pretrained = False
    cfg.fpn_encoder = "resnet18"
    cfg.pixels_per_image, cfg.mc_samples = 512, 3
    cfg.tta_scales = [1.0, 1.25]
    cfg.n_xai_images, cfg.occlusion_patch, cfg.n_robust_images = 3, 32, 4
    cfg.amp = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    cfg.amp = False
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

for sub in ["splits", "runs", "results", "figures", "tables"]:
    os.makedirs(os.path.join(cfg.project_dir, sub), exist_ok=True)
if not TRAIN_ONLY:   # the parallel training-only notebooks leave shared files alone
    with open(os.path.join(cfg.project_dir, "config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=2)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(0)
print("SMOKE_TEST =", SMOKE_TEST, "| device =", DEVICE, "| AMP =", cfg.amp,
      f"({AMP_DTYPE})" if cfg.amp else "")

### 1.5 Publication toolkit: figures and tables for print

Every figure and table in this notebook goes through two functions, so they all follow the same rules.

**`publish_figure`** draws each figure **twice**:

| Layout | Size | Use |
|---|---|---|
| `horizontal` | 182 mm (7.16 in) wide, panels side by side; one-panel figures 140 mm (1.5 columns) wide with the legend at the side | Double-column figures, slides |
| `vertical` | 89 mm (3.5 in) wide, panels stacked | Single-column figures |

Figures are drawn at their **final printed size**, so 8 pt text stays 8 pt in print. Each layout is saved as **vector PDF** (fonts embedded) plus **PNG and TIFF at 600 dpi**, without cropping, so the printed width is exact. An automatic **layout check** then measures every piece of text and reports any two labels that **overlap** or any text **clipped** at the figure edge.

**`publish_table`** writes each table as CSV, **LaTeX with `booktabs`** and a full-precision raw CSV. Add `\usepackage{booktabs,adjustbox}` to your LaTeX preamble; a table wider than the text column is scaled down to fit, narrower tables are left unchanged. For very wide tables in a two-column journal, change `table` to `table*`.

**Fixed numbering.** Figure and table numbers are defined once, in the order they appear in the paper, so re-running a cell never renumbers anything. Supplementary items are numbered S1, S2 and so on.

**Readable names.** Class labels such as `ACrack` or `WConccor` are shown in figures and tables as plain English names (Alligator crack, Washouts/concrete corrosion); Table 1 lists both.

**Colour and print.** Series keep one fixed colour each, and also differ by marker, line style or hatching, so figures stay readable when printed in greyscale.

In [ ]:
# CELL 05 · Publication toolkit: sizes, names, layout check, PDF/PNG/TIFF and LaTeX/Word export
# ---------------- physical sizes and formats ----------------
COL1, COL15, COL2 = 3.5, 5.51, 7.16   # single, 1.5 and double column widths in inches (89, 140, 182 mm)
FIG_DPI = 600
FIG_FORMATS = ("pdf", "png", "tiff")
STRICT_LAYOUT = False            # True: stop with an error if a figure has overlapping or clipped text
SHOW_BOTH_LAYOUTS = False        # True: also display the vertical layout in the notebook

FIG_DIR = os.path.join(cfg.project_dir, "figures")
TAB_DIR = os.path.join(cfg.project_dir, "tables")

def _pick_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"]:
        if name in have:
            return name
    return "DejaVu Sans"
FONT = _pick_font()

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, GRID, EDGE = "#0b0b0b", "#52514e", "#e4e3df", "#8a8984"
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
LINESTYLES = ["-", "--", "-.", ":"]
HATCHES = ["", "////", "....", "xxxx", "\\\\\\\\", "++"]

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [FONT, "DejaVu Sans"],
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "legend.title_fontsize": 7,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 4,
    "axes.linewidth": 0.6, "axes.edgecolor": EDGE, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.5,
    "lines.linewidth": 1.3, "lines.markersize": 3.5, "legend.frameon": False, "hatch.linewidth": 0.5,
    "figure.dpi": 110, "savefig.dpi": FIG_DPI, "savefig.bbox": None, "savefig.facecolor": "white",
    "figure.constrained_layout.use": True, "figure.facecolor": "white",
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-6, 6),   # plain numbers, no "1e-3" offsets
})

# ---------------- display names (no internal codes in print) ----------------
CLASS_NAMES = {
    "Crack": "Crack", "ACrack": "Alligator crack", "Wetspot": "Wet spot", "Efflorescence": "Efflorescence",
    "Rust": "Rust", "Rockpocket": "Rock pocket", "Hollowareas": "Hollow area", "Cavity": "Cavity",
    "Spalling": "Spalling", "Graffiti": "Graffiti", "Weathering": "Weathering",
    "Restformwork": "Residual formwork", "ExposedRebars": "Exposed rebars", "Bearing": "Bearing",
    "EJoint": "Expansion joint", "Drainage": "Drainage", "PEquipment": "Protective equipment",
    "JTape": "Joint tape", "WConccor": "Washouts/concrete corrosion"}
CLASS_GROUP = {c: "Concrete defect" for c in ["Crack", "ACrack", "Efflorescence", "Rockpocket", "WConccor",
                                              "Hollowareas", "Spalling", "Restformwork", "Cavity"]}
CLASS_GROUP.update({c: "General defect" for c in ["Wetspot", "Rust", "Graffiti", "Weathering"]})
CLASS_GROUP.update({c: "Object" for c in ["ExposedRebars", "Bearing", "EJoint", "Drainage", "PEquipment", "JTape"]})
assert set(CLASS_NAMES) == set(CLASSES) == set(CLASS_GROUP)

def _encoder_name(e):
    return {"efficientnet-b4": "EfficientNet-B4", "resnet18": "ResNet-18"}.get(e, e)
MODEL_KEYS = ["fpn_b4", "segformer", "dino_lora", "dino_full"]
MODEL_NAMES = {"fpn_b4": f"FPN ({_encoder_name(cfg.fpn_encoder)})",
               "segformer": "SegFormer-" + cfg.segformer_name.split("-")[-1].upper() if "mit-" in cfg.segformer_name else "SegFormer",
               "dino_lora": "DINOv2-B + LoRA", "dino_full": "DINOv2-B (full fine-tuning)"}
METHOD_KEYS = ["single", "mc", "tta", "ensemble"]
METHOD_NAMES = {"single": "Single model", "mc": "MC dropout", "tta": "Test-time augmentation", "ensemble": "Deep ensemble"}
CORRUPTION_KEYS = ["blur", "noise", "dark", "jpeg"]
CORRUPTION_NAMES = {"blur": "Gaussian blur", "noise": "Gaussian noise", "dark": "Low light", "jpeg": "JPEG compression"}
XAI_KEYS = ["gradcam", "occlusion", "random"]
XAI_NAMES = {"gradcam": "Seg-Grad-CAM", "occlusion": "Occlusion", "random": "Random baseline"}

def style_of(kind, key):
    """Fixed colour, marker, line style and hatch per entity, so each series looks the same in every figure."""
    order = {"model": MODEL_KEYS, "method": METHOD_KEYS, "corruption": CORRUPTION_KEYS, "xai": XAI_KEYS}[kind]
    i = order.index(key)
    return dict(color=PALETTE[i], marker=MARKERS[i], linestyle=LINESTYLES[i % 4], hatch=HATCHES[i])

# ---------------- fixed numbering of every figure and table ----------------
FIGURES = {
    "overview":     ("Figure 1", "Overview of the framework. The segmentation model returns multi-label defect masks together with a calibrated uncertainty map and a faithfulness-tested explanation map, so that the inspector can review doubtful regions first. Shaded boxes mark the contribution of this study."),
    "class_dist":   ("Figure 2", "Class distribution in the dacl10k training split: (a) number of training images containing each class and (b) approximate share of image area covered by each class."),
    "examples":     ("Figure 3", "Example training images with their polygon annotations. Colours identify classes as given in the legend; overlapping polygons show the multi-label nature of the task."),
    "per_class":    ("Figure 4", "Per-class intersection over union (IoU) on the test set for the four models ({seeds_mean})."),
    "reliability":  ("Figure 5", "Reliability diagrams after temperature scaling for (a) {c1}, (b) {c2} and (c) {c3}, {model}. Points on the dashed diagonal indicate perfect calibration."),
    "error_recall": ("Figure 6", "Uncertainty-guided review, {model}: share of all pixel errors found when the inspector reviews the given share of pixels in order of decreasing uncertainty. The dashed diagonal is random review; vertical lines mark the 5%, 10% and 20% review budgets."),
    "xai_gallery":  ("Figure 7", "Explanations for confusable defect classes, {model}. For each example: the image with the ground-truth outline of the target class, the predicted probability, the Seg-Grad-CAM map and the occlusion map (brighter = stronger attribution)."),
    "confusion":    ("Figure 8", "Class confusion on the test set, {model}. Diagonal: share of the ground-truth pixels of each class (rows) that are predicted as that class (recall). Off-diagonal: share predicted as another class (columns) where that class is not also labelled, so co-occurring labels, such as exposed rebars inside spalling, are not counted as confusion."),
    "robustness":   ("Figure 9", "Robustness to image corruption, {model}: (a) test mIoU and (b) mean epistemic uncertainty (MC dropout) at three corruption severities; severity 0 is the clean image."),
    "leakage":      ("Figure S1", "The closest near-duplicate image pairs found between the official training split and the test split (perceptual-hash Hamming distance d)."),
    "curves":       ("Figure S2", "Training curves (seed {seed}): (a) training loss and (b) validation mIoU at a threshold of 0.5."),
    "epochs":       ("Figure S3", "Training-length check, {model} (seed {seed}): (a) training loss and (b) validation mIoU at a threshold of 0.5 for the {short}-epoch and {long}-epoch schedules. The dotted line marks the end of the {short}-epoch schedule."),
}
TABLES = {
    "classes":      ("Table 1", "The 19 dacl10k classes: dataset label, name used in this paper, group, number of official training and test images containing the class, number of polygons in both splits and approximate share of image area in the training split."),
    "models":       ("Table 2", "Model configurations. Trainable parameters exclude frozen backbone weights."),
    "accuracy":     ("Table 3", "Segmentation accuracy (mIoU; {seeds_sd}). Per-class thresholds were tuned on the internal validation split. The leakage-free test set excludes test images with a near-duplicate in the training data."),
    "per_class":    ("Table 4", "Per-class test IoU ({seeds_mean})."),
    "bootstrap":    ("Table 5", "Paired bootstrap comparisons of test mIoU: each model against the replicated baseline {baseline}, and the best model on validation ({best}) against each other model ({n_boot} resamples of test images, percentile 95% confidence intervals; {seeds_mean}). ΔmIoU is the first model minus the second. P-values are Holm-adjusted within each test subset. The resampling covers test-image variability only, not training randomness."),
    "calibration":  ("Table 6", "Calibration of the uncertainty methods on the test set, {model}. ECE = expected calibration error (15 bins); ECE (active) is restricted to pixels that are labelled positive or predicted with p ≥ 0.05; NLL = negative log-likelihood. Lower is better for all calibration metrics."),
    "error_review": ("Table 7", "Uncertainty-guided error review, {model}. AURC = area under the risk–coverage curve (lower is better); the other columns give the share of pixel errors found when reviewing the most uncertain 5%, 10% or 20% of pixels."),
    "xai":          ("Table 8", "Faithfulness of the explanation methods over {n_images} test images, {model}. Deletion AUC: lower is better; insertion AUC: higher is better; rank correlation with a randomised model: lower is better. P-values: one-sided Wilcoxon signed-rank test against the random baseline."),
    "robustness":   ("Table 9", "Test mIoU and mean uncertainty (MC dropout) under image corruption, {model}, on {n_images} test images."),
    "splits":       ("Table S1", "Data splits used in this study. The near-duplicate audit found {n_pairs} image pair(s) with a perceptual-hash Hamming distance of at most {d} between the official training and validation splits."),
    "temperatures": ("Table S2", "Fitted per-class temperatures, {model}. T > 1 means the model was over-confident; T = 1 marks classes with too few positive pixels to fit."),
    "resolution":   ("Table S3", "Effect of input resolution on per-class test IoU, {model}, scored on {size} px ground truth."),
    "xai_leakage":  ("Table S4", "Off-target attribution leakage by target class (share of positive attribution outside the explained region), {model}."),
    "epochs":       ("Table S5", "Effect of training length, {model} (seed {seed}): the {short}-epoch protocol used for all models against a {long}-epoch schedule without early stopping. The last row gives the paired bootstrap difference ({n_boot} resamples of test images) with its 95% confidence interval.{sd_note}"),
}

PUB_PATH = os.path.join(cfg.project_dir, "results", "published.json")
PUBLISHED = json.load(open(PUB_PATH)) if os.path.exists(PUB_PATH) else {"figures": {}, "tables": {}}

def _short(label):            # "Figure 1" -> "Fig01", "Table S2" -> "TabS2"
    kind, num = label.split()
    pre = "Fig" if kind == "Figure" else "Tab"
    return pre + (num if num.startswith("S") else f"{int(num):02d}")

def _save_registry():
    with open(PUB_PATH, "w") as f:
        json.dump(PUBLISHED, f, indent=1)

# ---------------- layout check ----------------
def _visible_texts(fig):
    out = []
    def add(t, kind):
        if t is not None and t.get_visible() and t.get_text().strip():
            out.append((t, kind))
    for ax in fig.axes:
        if not ax.get_visible():
            continue
        for t in (ax.title, ax._left_title, ax._right_title):
            add(t, "title")
        add(ax.xaxis.label, "axis label"); add(ax.yaxis.label, "axis label")
        add(ax.xaxis.offsetText, "axis offset"); add(ax.yaxis.offsetText, "axis offset")
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            lo, hi = sorted(lim)
            tol = (hi - lo) * 1e-6 + 1e-12
            for tick in axis.get_major_ticks():
                if lo - tol <= tick.get_loc() <= hi + tol:
                    for lab in (tick.label1, tick.label2):
                        if lab.get_visible():
                            add(lab, "tick label")
        for t in ax.texts:
            add(t, "text")
        leg = ax.get_legend()
        if leg is not None:
            for t in leg.get_texts():
                add(t, "legend")
    for t in fig.texts:
        add(t, "text")
    for leg in fig.legends:
        for t in leg.get_texts():
            add(t, "legend")
    return out

def check_layout(fig, tol_pt=0.6):
    """Reports overlapping text, rotated text that cannot be checked, text clipped at the edge,
    and text that does not fit inside its diagram box."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    px_per_pt = fig.dpi / 72.0
    tol = tol_pt * px_per_pt
    items, issues = [], []
    fb = fig.bbox
    for t, kind in _visible_texts(fig):
        bb = t.get_window_extent(r)
        if bb.width <= 0 or bb.height <= 0:
            continue
        rot = round(t.get_rotation()) % 180
        label = t.get_text().replace("\n", " ")[:40]
        if rot not in (0, 90):
            issues.append(f"rotated {kind} ({rot} deg) cannot be checked: '{label}'")
        if bb.x0 < fb.x0 - tol or bb.y0 < fb.y0 - tol or bb.x1 > fb.x1 + tol or bb.y1 > fb.y1 + tol:
            issues.append(f"{kind} clipped at the figure edge: '{label}'")
        items.append((label, kind, bb))
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            a, b = items[i][2], items[j][2]
            ox = min(a.x1, b.x1) - max(a.x0, b.x0)
            oy = min(a.y1, b.y1) - max(a.y0, b.y0)
            if ox > tol and oy > tol:
                issues.append(f"overlap: {items[i][1]} '{items[i][0]}' and {items[j][1]} '{items[j][0]}'")
    for ax in fig.axes:                          # repeated tick labels (e.g. "0.001, 0.001") are misleading
        for name, axis, lim in (("x", ax.xaxis, ax.get_xlim()), ("y", ax.yaxis, ax.get_ylim())):
            lo, hi = sorted(lim)
            labs = [tk.label1.get_text() for tk in axis.get_major_ticks()
                    if tk.label1.get_visible() and tk.label1.get_text().strip() and lo <= tk.get_loc() <= hi]
            dup = sorted({l for l in labs if labs.count(l) > 1})
            if dup:
                issues.append(f"repeated tick labels on the {name} axis: {', '.join(dup[:3])}")
    for t, patch in getattr(fig, "_contain_checks", []):
        tb, pb = t.get_window_extent(r), patch.get_window_extent(r)
        if tb.x0 < pb.x0 + tol or tb.x1 > pb.x1 - tol or tb.y0 < pb.y0 + tol or tb.y1 > pb.y1 - tol:
            issues.append(f"text does not fit inside its box: '{t.get_text()[:40]}'")
    return issues

# ---------------- figures ----------------
def fit_height(fig, pad_in=0.04, max_iter=4):
    """Keeps the printed width fixed and trims empty bands above and below the content."""
    eng = fig.get_layout_engine()
    if eng is None or type(eng).__name__ != "ConstrainedLayoutEngine":
        return
    for _ in range(max_iter):
        fig.canvas.draw()
        tb = fig.get_tightbbox(fig.canvas.get_renderer())
        w, h = fig.get_size_inches()
        slack = (tb.y0 - pad_in) + (h - tb.y1 - pad_in)
        if slack < 0.03:
            break
        fig.set_size_inches(w, h - slack)

def wrap_label(text, width):
    """Wraps at spaces or after a slash, never inside a word."""
    tokens = []
    for word in text.split(" "):
        parts = [p for p in re.split(r"(?<=/)", word) if p]
        tokens += [(p, k < len(parts) - 1) for k, p in enumerate(parts)]
    lines, cur, glue = [], "", False
    for tok, ends_with_slash in tokens:
        sep = "" if (glue or not cur) else " "
        if cur and len(cur) + len(sep) + len(tok) > width:
            lines.append(cur); cur = tok
        else:
            cur += sep + tok
        glue = ends_with_slash
    if cur:
        lines.append(cur)
    return "\n".join(lines)

def crop_to_aspect(img, aspect=4 / 3):
    """Centre crop so that every photo in a grid has the same shape and panels line up."""
    h, w = img.shape[:2]
    if w / h > aspect:
        nw = int(round(h * aspect)); x0 = (w - nw) // 2
        return img[:, x0:x0 + nw]
    nh = int(round(w / aspect)); y0 = (h - nh) // 2
    return img[y0:y0 + nh]

def publish_figure(key, draw, layouts=("horizontal", "vertical"), **caption_vars):
    """draw(layout) must return a matplotlib figure drawn at its final printed size."""
    if TRAIN_ONLY:   # part notebooks run in parallel: never write shared figure files from two sessions
        print(f"{FIGURES[key][0]}: not produced in this training-only notebook (the results notebook makes it).")
        return None
    label, caption = FIGURES[key]
    caption = caption.format(**caption_vars)
    stem = f"{_short(label)}_{key}"
    info = {"label": label, "caption": caption, "files": {}, "size_in": {}, "issues": {}}
    for k, layout in enumerate(layouts):
        fig = draw(layout)
        fit_height(fig)
        issues = check_layout(fig)
        info["issues"][layout] = issues
        if issues:
            print(f"Layout check, {label} ({layout}): {len(issues)} issue(s)")
            for s in issues[:10]:
                print("   -", s)
            if STRICT_LAYOUT:
                plt.close(fig)
                raise RuntimeError(f"{label} ({layout}) failed the layout check (STRICT_LAYOUT = True).")
        paths = []
        png = os.path.join(FIG_DIR, f"{stem}_{layout}.png")
        for ext in FIG_FORMATS:
            p = os.path.join(FIG_DIR, f"{stem}_{layout}.{ext}")
            if ext == "tiff":
                fig.savefig(png, dpi=FIG_DPI) if "png" not in FIG_FORMATS else None
                Image.open(png).convert("RGB").save(p, compression="tiff_lzw", dpi=(FIG_DPI, FIG_DPI))
            else:
                fig.savefig(p, dpi=FIG_DPI)
            paths.append(p)
        w, h = fig.get_size_inches()
        info["files"][layout] = paths
        info["size_in"][layout] = [round(float(w), 2), round(float(h), 2)]
        if k == 0 or SHOW_BOTH_LAYOUTS:
            plt.show()
        else:
            plt.close(fig)
    PUBLISHED["figures"][key] = info
    _save_registry()
    ok = all(not v for v in info["issues"].values())
    sizes = ", ".join(f"{l} {s[0]:.2f} x {s[1]:.2f} in" for l, s in info["size_in"].items())
    display(Markdown(f"**{label}.** {caption}  \n*Saved as {stem}_[horizontal|vertical].[pdf|png|tiff] "
                     f"({sizes}; {FIG_DPI} dpi).* Layout check: **{'passed' if ok else 'see warnings above'}**."))
    return info

def legend_below(fig, handles, labels, ncol=None):
    w = fig.get_size_inches()[0]
    ncol = ncol or (4 if w > 5 else 2)
    return fig.legend(handles, labels, loc="outside lower center", ncol=min(ncol, len(labels)),
                      handlelength=2.2, columnspacing=1.2, borderaxespad=0.2)

def series_handle(kind, key, name, bar=False):
    st = style_of(kind, key)
    if bar:
        return Patch(facecolor=st["color"], hatch=st["hatch"], edgecolor="white", linewidth=0.4, label=name)
    return Line2D([], [], color=st["color"], marker=st["marker"], linestyle=st["linestyle"], label=name)

def image_axes(ax, img, title=None, ylabel=None, title_loc="center", bold=False, cmap=None, vmin=None, vmax=None):
    ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, loc=title_loc, fontweight="bold" if bold else "normal")
    if ylabel:
        ax.set_ylabel(ylabel)

def thumbnail(img, max_side=1000):
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img

# ---------------- tables ----------------
def f3(x, d=3):
    return "–" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"

def pm(mean, sd, d=3):
    if sd is None or not np.isfinite(sd):
        return f3(mean, d)
    return f"{mean:.{d}f} ± {sd:.{d}f}"

def pfmt(p):
    if p is None or not np.isfinite(p):
        return "–"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

_TEX_ESC = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#", "_": r"\_",
            "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
_TEX_UNI = {"±": r"$\pm$", "Δ": r"$\Delta$", "×": r"$\times$", "≤": r"$\leq$", "≥": r"$\geq$", "–": "--",
            "—": "---", "↑": r"$\uparrow$", "↓": r"$\downarrow$", "→": r"$\rightarrow$", "<": r"$<$", ">": r"$>$"}

def tex(s):
    return "".join(_TEX_ESC.get(ch) or _TEX_UNI.get(ch) or ch for ch in str(s))

def _is_numeric_col(values):
    return all(re.fullmatch(r"[\s<>≥≤±+\-–.,\[\]\d%]+", str(v)) for v in values)

def to_booktabs(df, label, caption, key):
    cols = list(df.columns)
    align = "".join("r" if _is_numeric_col(df[c]) else "l" for c in cols)
    lines = [r"\begin{table}[tbp]", r"\centering", r"\small",
             rf"\caption{{{tex(caption)}}}", rf"\label{{tab:{key}}}",
             r"\begin{adjustbox}{max width=\linewidth}",
             rf"\begin{{tabular}}{{{align}}}", r"\toprule",
             " & ".join(tex(c) for c in cols) + r" \\", r"\midrule"]
    for row in df.itertuples(index=False):
        lines.append(" & ".join(tex(v) for v in row) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{adjustbox}", r"\end{table}"]
    return "\n".join(lines)

def publish_table(key, df, raw=None, **caption_vars):
    """df: display-ready DataFrame (strings). raw: optional full-precision DataFrame."""
    if TRAIN_ONLY:
        print(f"{TABLES[key][0]}: not produced in this training-only notebook (the results notebook makes it).")
        return None
    label, caption = TABLES[key]
    caption = caption.format(**caption_vars)
    stem = f"{_short(label)}_{key}"
    df = df.reset_index(drop=True).astype(str)
    df.to_csv(os.path.join(TAB_DIR, f"{stem}.csv"), index=False)
    if raw is not None:
        raw.to_csv(os.path.join(TAB_DIR, f"{stem}_raw.csv"))
    with open(os.path.join(TAB_DIR, f"{stem}.tex"), "w") as f:
        f.write(to_booktabs(df, label, caption, key))
    PUBLISHED["tables"][key] = {"label": label, "caption": caption, "columns": list(df.columns),
                                "rows": df.values.tolist(), "stem": stem}
    _save_registry()
    display(Markdown(f"**{label}.** {caption}"))
    try:
        display(df.style.hide(axis="index"))
    except Exception:
        display(df)
    print(f"Saved {stem}.csv and {stem}.tex in {TAB_DIR}")

# ---------------- Word report with every table and figure in numbered order ----------------
def join_and(items):
    items = [str(i) for i in items]
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]

def seeds_text(n):
    """Caption wording that reads correctly for one or several seeds."""
    if n <= 1:
        return {"seeds_sd": "one training run per model", "seeds_mean": "one training run per model"}
    return {"seeds_sd": f"mean ± standard deviation over {n} seeds", "seeds_mean": f"mean over {n} seeds"}

def _order_key(label):
    kind, num = label.split()
    supp = num.startswith("S")
    return (supp, 0 if kind == "Table" else 1, int(num.lstrip("S")))

def build_word_report(path):
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    doc = Document()
    sec = doc.sections[0]
    sec.page_width, sec.page_height = Inches(8.27), Inches(11.69)
    for side in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
        setattr(sec, side, Inches(0.79))
    usable = 8.27 - 2 * 0.79
    normal = doc.styles["Normal"]; normal.font.name = "Arial"; normal.font.size = Pt(10)
    def caption_par(label, text, before=6, after=4):
        p = doc.add_paragraph()
        p.paragraph_format.space_before, p.paragraph_format.space_after = Pt(before), Pt(after)
        r = p.add_run(label + ". "); r.bold = True; r.font.size = Pt(9)
        r2 = p.add_run(text); r2.font.size = Pt(9)
        return p
    def shade(cell, fill="EAF1F8"):
        tcPr = cell._tc.get_or_add_tcPr(); shd = OxmlElement("w:shd")
        shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), fill); tcPr.append(shd)
    items = [("table", k, v) for k, v in PUBLISHED["tables"].items()] + \
            [("figure", k, v) for k, v in PUBLISHED["figures"].items()]
    items.sort(key=lambda it: _order_key(it[2]["label"]))
    title = doc.add_paragraph(); rt = title.add_run("Figures and tables"); rt.bold = True; rt.font.size = Pt(14)
    doc.add_paragraph("dacl10k bridge damage segmentation: results generated by the analysis notebook.").runs[0].font.size = Pt(9)
    current = None
    for kind, key, info in items:
        section = ("Supplementary " if _order_key(info["label"])[0] else "") + ("tables" if kind == "table" else "figures")
        if section != current:
            h = doc.add_paragraph(); hr = h.add_run(section[0].upper() + section[1:]); hr.bold = True; hr.font.size = Pt(12)
            h.paragraph_format.space_before = Pt(12); current = section
        if kind == "table":
            cap = caption_par(info["label"], info["caption"])
            cap.paragraph_format.keep_with_next = True
            cols, rows = info["columns"], info["rows"]
            t = doc.add_table(rows=1 + len(rows), cols=len(cols)); t.style = "Table Grid"
            t.autofit = False
            numeric = [_is_numeric_col([r[j] for r in rows]) if rows else False for j in range(len(cols))]
            CH = 0.075                                    # inches per character at 8 pt
            min_w, pref = [], []
            for j, c in enumerate(cols):
                vals = [str(r[j]) for r in rows]
                hdr_word = max([len(w) for w in str(c).split()] + [1]) * 1.2      # header is bold
                val_word = max([len(w) for v in vals for w in v.split()] + [1])
                longest_val = max([len(v) for v in vals] + [1])
                pad = 0.22                                                         # cell margins
                # numbers never wrap; text may wrap at spaces, never inside a word
                min_w.append(max(hdr_word, val_word, longest_val if numeric[j] else 0) * CH + pad)
                pref.append(max(len(str(c)) * 0.7, longest_val) * CH + pad)
            if sum(pref) <= usable:
                widths = [w * usable / sum(pref) for w in pref]
            elif sum(min_w) <= usable:
                grow = [max(pw - mw, 0) for pw, mw in zip(pref, min_w)]
                extra = usable - sum(min_w)
                widths = [mw + (extra * g / sum(grow) if sum(grow) else 0) for mw, g in zip(min_w, grow)]
            else:
                widths = [w * usable / sum(min_w) for w in min_w]
            layout = OxmlElement("w:tblLayout"); layout.set(qn("w:type"), "fixed"); t._tbl.tblPr.append(layout)
            for j, col in enumerate(t.columns):
                col.width = Inches(widths[j])
            for row in t.rows:
                trPr = row._tr.get_or_add_trPr()
                cs = OxmlElement("w:cantSplit"); trPr.append(cs)
                for j, cell in enumerate(row.cells):
                    cell.width = Inches(widths[j])
            hdr = t.rows[0]._tr.get_or_add_trPr(); th = OxmlElement("w:tblHeader"); hdr.append(th)
            for j, c in enumerate(cols):
                cell = t.rows[0].cells[j]; cell.text = ""; run = cell.paragraphs[0].add_run(str(c))
                run.bold = True; run.font.size = Pt(8); shade(cell)
            for i, row in enumerate(rows, start=1):
                for j, v in enumerate(row):
                    cell = t.rows[i].cells[j]; cell.text = ""
                    par = cell.paragraphs[0]; run = par.add_run(str(v)); run.font.size = Pt(8)
                    par.paragraph_format.keep_with_next = i < len(rows)
                    if numeric[j]:
                        par.alignment = WD_ALIGN_PARAGRAPH.RIGHT
            for cell in t.rows[0].cells:
                cell.paragraphs[0].paragraph_format.keep_with_next = True
            doc.add_paragraph()
        else:
            png = next((p for p in info["files"].get("horizontal", []) if p.endswith(".png")), None)
            if png and os.path.exists(png):
                w = min(info["size_in"]["horizontal"][0], usable)
                doc.add_picture(png, width=Inches(w))
                doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
                doc.paragraphs[-1].paragraph_format.keep_with_next = True
            caption_par(info["label"], info["caption"], before=2, after=10)
    doc.save(path)
    return path

def write_index(path):
    lines = ["# Figures and tables", ""]
    items = sorted(list(PUBLISHED["tables"].values()) + list(PUBLISHED["figures"].values()),
                   key=lambda v: _order_key(v["label"]))
    for v in items:
        files = v.get("stem") or ", ".join(os.path.basename(p) for p in v["files"].get("horizontal", []))
        lines += [f"**{v['label']}.** {v['caption']}", f"*Files:* {files}", ""]
    open(path, "w").write("\n".join(lines))
    return path

print(f"Font: {FONT} | widths: {COL1} in (single column), {COL2} in (double column) | {FIG_DPI} dpi | formats: {FIG_FORMATS}")
print(f"{len(PUBLISHED['figures'])} figures and {len(PUBLISHED['tables'])} tables already published in earlier sessions.")

### 1.6 Figure 1: framework overview

A diagram of the study design, drawn with the same fonts and sizes as the data figures. The layout check also confirms that every label fits inside its box.

In [ ]:
# CELL 06 · Figure 1: framework diagram
def draw_overview(layout):
    accent_fill, plain_fill = "#dce9f7", "white"
    if layout == "horizontal":
        W, H = COL2, 2.55
        boxes = {
            "input":   (0.10, 0.83, 1.18, 0.90, ["Inspection image", "RGB photo of a", "concrete surface"], False),
            "model":   (1.70, 0.76, 1.42, 1.04, ["Segmentation model", "DINOv2-B + LoRA", "19 sigmoid outputs"], False),
            "masks":   (3.54, 1.78, 1.80, 0.64, ["Defect masks", "multi-label, 19 classes"], False),
            "unc":     (3.54, 0.96, 1.80, 0.64, ["Uncertainty map", "calibrated per class"], True),
            "xai":     (3.54, 0.14, 1.80, 0.64, ["Explanation map", "faithfulness-tested"], True),
            "review":  (5.76, 0.83, 1.30, 0.90, ["Inspector review", "checks flagged", "regions first"], False),
        }
    else:
        W, H = COL1, 4.75
        boxes = {
            "input":   (0.55, 3.86, 2.40, 0.72, ["Inspection image", "RGB photo of a concrete surface"], False),
            "model":   (0.45, 2.70, 2.60, 0.80, ["Segmentation model", "DINOv2-B + LoRA", "19 sigmoid outputs"], False),
            "masks":   (0.06, 1.30, 1.06, 0.98, ["Defect masks", "multi-label,", "19 classes"], False),
            "unc":     (1.22, 1.30, 1.06, 0.98, ["Uncertainty map", "calibrated", "per class"], True),
            "xai":     (2.38, 1.30, 1.06, 0.98, ["Explanation map", "faithfulness", "tested"], True),
            "review":  (0.55, 0.12, 2.40, 0.72, ["Inspector review", "checks flagged regions first"], False),
        }
    fig = plt.figure(figsize=(W, H), layout="none")
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")
    fig._contain_checks = []
    centers = {}
    for k, (x, y, w, h, lines, accent) in boxes.items():
        patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.06",
                               facecolor=accent_fill if accent else plain_fill,
                               edgecolor=PALETTE[0] if accent else EDGE, linewidth=1.0 if accent else 0.7)
        ax.add_patch(patch)
        n = len(lines); step = 0.135
        y0 = y + h / 2 + (n - 1) * step / 2
        for i, line in enumerate(lines):
            t = ax.text(x + w / 2, y0 - i * step, line, ha="center", va="center",
                        fontsize=7.5 if i == 0 else 6.5, fontweight="bold" if i == 0 else "normal",
                        color=INK if i == 0 else INK2)
            fig._contain_checks.append((t, patch))
        centers[k] = (x, y, w, h)
    arrow = dict(arrowstyle="-|>", color=EDGE, lw=0.8, shrinkA=0, shrinkB=0, mutation_scale=7)
    def right(k):  x, y, w, h = centers[k]; return (x + w, y + h / 2)
    def left(k):   x, y, w, h = centers[k]; return (x, y + h / 2)
    def top(k):    x, y, w, h = centers[k]; return (x + w / 2, y + h)
    def bottom(k): x, y, w, h = centers[k]; return (x + w / 2, y)
    if layout == "horizontal":
        ax.annotate("", xy=left("model"), xytext=right("input"), arrowprops=arrow)
        mx = (right("model")[0] + left("masks")[0]) / 2
        for k in ("masks", "unc", "xai"):
            ax.plot([right("model")[0], mx, mx], [right("model")[1], right("model")[1], left(k)[1]], color=EDGE, lw=0.8)
            ax.annotate("", xy=left(k), xytext=(mx, left(k)[1]), arrowprops=arrow)
        rx = (right("masks")[0] + left("review")[0]) / 2
        for k in ("masks", "unc", "xai"):
            ax.plot([right(k)[0], rx, rx], [right(k)[1], right(k)[1], left("review")[1]], color=EDGE, lw=0.8)
        ax.annotate("", xy=left("review"), xytext=(rx, left("review")[1]), arrowprops=arrow)
    else:
        ax.annotate("", xy=top("model"), xytext=bottom("input"), arrowprops=arrow)
        my = (bottom("model")[1] + top("masks")[1]) / 2
        for k in ("masks", "unc", "xai"):
            ax.plot([bottom("model")[0], bottom("model")[0], top(k)[0]], [bottom("model")[1], my, my], color=EDGE, lw=0.8)
            ax.annotate("", xy=top(k), xytext=(top(k)[0], my), arrowprops=arrow)
        ry = (bottom("masks")[1] + top("review")[1]) / 2
        for k in ("masks", "unc", "xai"):
            ax.plot([bottom(k)[0], bottom(k)[0], top("review")[0]], [bottom(k)[1], ry, ry], color=EDGE, lw=0.8)
        ax.annotate("", xy=top("review"), xytext=(top("review")[0], ry), arrowprops=arrow)
    return fig

publish_figure("overview", draw_overview)

---
## 2. Getting the data

The dataset is fetched automatically from the first source that works, in this order:

| # | Source | Notes |
|---|---|---|
| A | **Official AWS archive** `dacl10k_v2_devphase.zip`, listed in the [dacl10k-toolkit README](https://github.com/phiyodr/dacl10k-toolkit) | Verified with the published SHA-256 checksum. This link stopped working in November 2026 (HTTP error), so the cell falls through to B. |
| B | **Hugging Face mirror** [`Voxel51/dacl10k`](https://huggingface.co/datasets/Voxel51/dacl10k) (about 5.2 GB, FiftyOne format) | The same 8,922 labelled-split images with the original file names. Its polygon labels are converted here into the official folder layout and JSON format, so the rest of the notebook is unchanged. |
| C | **Your own copy on Google Drive** | If you obtain `dacl10k_v2_devphase.zip` elsewhere (for example from the dataset authors), put it at `PROJECT_DIR/dacl10k_v2_devphase.zip` and re-run. It is used before A and B. |

Whichever source is used, the result is the official layout:

```
annotations/train       (6,935 JSON)     images/train          (6,935 JPG)
annotations/validation  (975 JSON)       images/validation     (975 JPG)
                                         images/testdev        (1,012 JPG, no labels)
```

The cell prints which source was used. **Record it in your paper**: if you use the mirror, cite it alongside the original dataset paper, and note that the conversion was checked against the expected split sizes and class names (both are verified below and in Section 3).

**Important for the research design:** labels are public only for `train` and `validation`; the test splits are scored on the organisers' server. This notebook therefore uses the official **validation split as its held-out test set** and carves a new internal validation split out of the training data (Section 5). Say this explicitly in the paper's methods section.

In smoke-test mode the download is replaced by a generator that writes a tiny dataset in exactly the same format.

In [ ]:
# CELL 07 · Download and prepare dacl10k
DACL_URL = "https://dacl10k.s3.eu-central-1.amazonaws.com/dacl10k-challenge/dacl10k_v2_devphase.zip"
DACL_SHA256 = "dcbcd5fb82699076a2c7f3a72492a9ef798870e0ca1f0c9399360f273ea95260"
HF_REPO = "Voxel51/dacl10k"
SPLIT_RE = re.compile(r"dacl10k_v\d+_(train|validation|testdev|testchallenge)_\d+", re.I)

# Label spellings that may differ between sources -> official toolkit names
LABEL_ALIASES = {"alligatorcrack": "ACrack", "exposedrebars": "ExposedRebars", "exposedrebar": "ExposedRebars",
                 "expansionjoint": "EJoint", "protectiveequipment": "PEquipment", "jointtape": "JTape",
                 "washoutsconcretecorrosion": "WConccor", "concretecorrosion": "WConccor", "concretec": "WConccor",
                 "hollowarea": "Hollowareas", "rockpockets": "Rockpocket"}
_CANON = {re.sub(r"[^a-z]", "", c.lower()): c for c in CLASSES}

def canonical_label(label):
    key = re.sub(r"[^a-z]", "", str(label).lower())
    return _CANON.get(key) or LABEL_ALIASES.get(key) or str(label)

def sha256sum(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def dataset_ready(root):
    return (len(glob.glob(os.path.join(root, "annotations", "train", "*.json"))) > 0
            and len(glob.glob(os.path.join(root, "images", "train", "*.jpg"))) > 0)

def extract_official_zip(zip_path, root, verify=True):
    if verify:
        print("Verifying checksum (takes a minute)...")
        digest = sha256sum(zip_path)
        if digest != DACL_SHA256:
            print(f"  Checksum mismatch ({digest[:16]}...). The file is not the official archive or is incomplete.")
            return False
        print("  Checksum OK.")
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(root)
    if not os.path.isdir(os.path.join(root, "annotations")):   # archive with a top-level folder
        inner = [d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(os.path.join(d, "annotations"))]
        if inner:
            for item in os.listdir(inner[0]):
                shutil.move(os.path.join(inner[0], item), root)
    return dataset_ready(root)

def try_drive_zip(root):
    zip_path = os.path.join(cfg.project_dir, "dacl10k_v2_devphase.zip")
    if not os.path.exists(zip_path):
        return False
    print("[C] Found your copy on Drive:", zip_path)
    return extract_official_zip(zip_path, root)

def try_aws(root):
    import requests
    print("[A] Trying the official AWS archive...")
    try:
        r = requests.get(DACL_URL, stream=True, timeout=30)
    except Exception as e:
        print("  Not reachable:", type(e).__name__); return False
    if r.status_code != 200:
        print(f"  Server answered HTTP {r.status_code} ({r.reason}); the link is no longer available."); return False
    zip_path = os.path.join(root, "dacl10k_v2_devphase.zip")
    total = int(r.headers.get("content-length", 0))
    with open(zip_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="AWS download") as bar:
        for chunk in r.iter_content(1 << 22):
            f.write(chunk); bar.update(len(chunk))
    ok = extract_official_zip(zip_path, root)
    os.remove(zip_path)
    return ok

def fiftyone_shapes(points):
    """FiftyOne Polyline.points is a list of shapes, each a list of [x, y] in relative [0, 1] coordinates.
    Accepts a single shape too."""
    if not points:
        return []
    if isinstance(points[0][0], (int, float)):
        return [points]
    return points

def convert_fiftyone_export(src, root):
    """Converts a FiftyOne dataset export (samples.json + images) into the official dacl10k layout."""
    data = json.load(open(os.path.join(src, "samples.json")))
    samples = data["samples"] if isinstance(data, dict) else data
    print(f"  {len(samples)} samples in samples.json")
    poly_field = None
    for cand in ["semseg_ground_truth_filled", "semseg_ground_truth", "ground_truth", "polylines"]:
        if any(isinstance(s.get(cand), dict) and s[cand].get("polylines") for s in samples[:200]):
            poly_field = cand; break
    if poly_field is None:
        raise RuntimeError("No polyline label field found in samples.json; fields seen: " + ", ".join(sorted(samples[0].keys())))
    print("  Using label field:", poly_field)
    counts, label_counts, n_coords_abs = {}, {}, 0
    for s in tqdm(samples, desc="Converting"):
        fp = s["filepath"]
        src_img = fp if os.path.isabs(fp) and os.path.exists(fp) else os.path.join(src, fp)
        name = os.path.basename(fp)
        m = SPLIT_RE.search(name)
        split = m.group(1).lower() if m else next((t for t in s.get("tags", []) if t in ("train", "validation", "testdev", "testchallenge")), None)
        if split is None:
            continue
        md_ = s.get("metadata") or {}
        w, h = md_.get("width"), md_.get("height")
        if not w or not h:
            with Image.open(src_img) as im:
                w, h = im.size
        os.makedirs(os.path.join(root, "images", split), exist_ok=True)
        dst_img = os.path.join(root, "images", split, name)
        if not os.path.exists(dst_img):
            shutil.move(src_img, dst_img)
        counts[split] = counts.get(split, 0) + 1
        if split not in ("train", "validation"):
            continue
        shapes = []
        for pl in (s.get(poly_field) or {}).get("polylines", []):
            label = canonical_label(pl.get("label"))
            for shape in fiftyone_shapes(pl.get("points")):
                pts = np.asarray(shape, dtype=np.float64)
                if len(pts) < 3:
                    continue
                if pts.max() > 1.5:          # already in pixels
                    n_coords_abs += 1
                else:
                    pts = pts * [w, h]
                shapes.append({"label": label, "points": pts.tolist(), "shape_type": "polygon"})
                label_counts[label] = label_counts.get(label, 0) + 1
        os.makedirs(os.path.join(root, "annotations", split), exist_ok=True)
        ann = {"imageWidth": int(w), "imageHeight": int(h), "imageName": name, "split": split,
               "dacl10k_version": "v2", "source": f"huggingface:{HF_REPO}", "shapes": shapes}
        with open(os.path.join(root, "annotations", split, name.replace(".jpg", ".json")), "w") as f:
            json.dump(ann, f)
    print("  Images per split:", counts)
    print("  Polygons per label:", dict(sorted(label_counts.items(), key=lambda kv: -kv[1])))
    unknown = sorted(set(label_counts) - set(CLASSES))
    if unknown:
        print("  WARNING: labels not in the 19 official classes (extend LABEL_ALIASES):", unknown)
    if n_coords_abs:
        print(f"  Note: {n_coords_abs} shapes were already in pixel coordinates and were kept as they are.")
    return dataset_ready(root)

def try_huggingface(root):
    print(f"[B] Trying the Hugging Face mirror {HF_REPO} (about 5.2 GB)...")
    try:
        from huggingface_hub import snapshot_download
        src = os.path.join(root, "_hf_mirror")
        snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=src, max_workers=8)
    except Exception as e:
        print("  Download failed:", type(e).__name__, str(e)[:300]); return False
    ok = convert_fiftyone_export(src, root)
    if ok:
        shutil.rmtree(src, ignore_errors=True)   # images were moved out; free the rest
    return ok

def download_dacl10k(root):
    if dataset_ready(root):
        print("Dataset already prepared at", root); return
    for source in (try_drive_zip, try_aws, try_huggingface):
        if source(root):
            print("Dataset ready.\n"); return
    raise RuntimeError(
        "Could not obtain dacl10k from any source.\n"
        f"  Option 1: download dacl10k_v2_devphase.zip yourself and place it at {cfg.project_dir}/\n"
        f"  Option 2: check that https://huggingface.co/datasets/{HF_REPO} is reachable, then re-run this cell.")

def make_synthetic_dacl(root, n_train=48, n_val=16, n_test=4, seed=0):
    """Writes a tiny dataset in dacl10k format. Each class is drawn in its own colour so a model can learn it."""
    if os.path.isdir(os.path.join(root, "annotations", "train")):
        print("Synthetic dataset already exists at", root); return
    rng = np.random.default_rng(seed)
    colours = rng.integers(30, 225, size=(C, 3))
    for split, n in [("train", n_train), ("validation", n_val), ("testdev", n_test), ("testchallenge", n_test)]:
        os.makedirs(os.path.join(root, "images", split), exist_ok=True)
        if split in ("train", "validation"):
            os.makedirs(os.path.join(root, "annotations", split), exist_ok=True)
        for i in range(n):
            w, h = int(rng.integers(320, 480)), int(rng.integers(240, 360))
            img = (rng.normal(128, 20, size=(h, w, 3))).clip(0, 255).astype(np.uint8)
            shapes = []
            for _ in range(int(rng.integers(1, 5))):
                c = int(rng.integers(0, C))  # all 19 classes, so every table and figure is exercised
                cx, cy, r = rng.uniform(40, w - 40), rng.uniform(40, h - 40), rng.uniform(15, 50)
                ang = np.sort(rng.uniform(0, 2 * np.pi, 7))
                pts = np.stack([cx + r * np.cos(ang), cy + r * np.sin(ang)], 1)
                cv2.fillPoly(img, [pts.astype(np.int32)], tuple(int(v) for v in colours[c]))
                shapes.append({"label": CLASSES[c], "points": pts.tolist(), "shape_type": "polygon"})
            name = f"dacl10k_v2_{split}_{i:04d}.jpg"
            cv2.imwrite(os.path.join(root, "images", split, name), img)
            if split in ("train", "validation"):
                ann = {"imageWidth": w, "imageHeight": h, "imageName": name, "split": split,
                       "dacl10k_version": "v2", "shapes": shapes}
                with open(os.path.join(root, "annotations", split, name.replace(".jpg", ".json")), "w") as f:
                    json.dump(ann, f)
    # Plant one cross-split near-duplicate so the leakage audit has something to find.
    src = "dacl10k_v2_train_0000"; dst = "dacl10k_v2_validation_0000"
    shutil.copy(os.path.join(root, "images", "train", src + ".jpg"), os.path.join(root, "images", "validation", dst + ".jpg"))
    ann = json.load(open(os.path.join(root, "annotations", "train", src + ".json")))
    ann["imageName"], ann["split"] = dst + ".jpg", "validation"
    json.dump(ann, open(os.path.join(root, "annotations", "validation", dst + ".json"), "w"))
    print("Synthetic dataset written to", root)

if SMOKE_TEST:
    make_synthetic_dacl(cfg.data_root)
else:
    download_dacl10k(cfg.data_root)

EXPECTED = {"train": 6935, "validation": 975, "testdev": 1012}
problems = []
for split in ["train", "validation", "testdev", "testchallenge"]:
    n_img = len(glob.glob(os.path.join(cfg.data_root, "images", split, "*.jpg")))
    n_ann = len(glob.glob(os.path.join(cfg.data_root, "annotations", split, "*.json")))
    note = ""
    if not SMOKE_TEST and split in EXPECTED and n_img != EXPECTED[split]:
        note = f"  <-- expected {EXPECTED[split]}"; problems.append(split)
    print(f"{split:14s} images={n_img:5d}  annotations={n_ann:5d}{note}")
if problems:
    print("\nSplit sizes differ from the official release; report the actual counts in the paper.")

---
## 3. Exploratory data analysis

Before modelling, we check four things the paper's methods section will need:

1. **Dataset version.** Every JSON carries a `dacl10k_version` field. Results must name the version, because v1 had 18 classes and v2 has 19.
2. **Label vocabulary.** Any label outside the 19 expected names would silently vanish from the masks, so we list them.
3. **Class frequency**, by images and by polygon area (**Table 1, Figure 2**). Area is computed with the shoelace formula on the polygons; overlapping polygons of one class are double-counted, so area shares are approximate.
4. **Image–annotation consistency.** Recorded width and height are compared with the actual JPEG size; a mismatch would misplace every polygon.

In [ ]:
# CELL 08 · Read annotations and check labels
def polygon_area(pts):
    p = np.asarray(pts, float)
    if len(p) < 3: return 0.0
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

records, shape_rows = [], []
for split in ["train", "validation"]:
    for path in sorted(glob.glob(os.path.join(cfg.data_root, "annotations", split, "*.json"))):
        a = json.load(open(path))
        name = os.path.basename(path).replace(".json", ".jpg")
        img_area = a["imageWidth"] * a["imageHeight"]
        labels = [s["label"] for s in a["shapes"]]
        records.append({"name": name, "split": split, "w": a["imageWidth"], "h": a["imageHeight"],
                        "version": a.get("dacl10k_version", "?"), "n_polygons": len(labels),
                        "labels": sorted(set(labels))})
        for s in a["shapes"]:
            shape_rows.append({"name": name, "split": split, "label": s["label"],
                               "shape_type": s.get("shape_type", "?"),
                               "area_frac": polygon_area(s["points"]) / img_area})

meta = pd.DataFrame(records)
shapes_df = pd.DataFrame(shape_rows)
print("Versions found:", meta["version"].value_counts().to_dict())
unknown = sorted(set(shapes_df["label"]) - set(CLASSES))
print("Labels outside the 19 classes:", unknown or "none")
print("Shape types:", shapes_df["shape_type"].value_counts().to_dict())
print(f"Polygons per image: mean {meta.n_polygons.mean():.1f}, median {meta.n_polygons.median():.0f}, max {meta.n_polygons.max()}")
print(f"Image size: median {meta.w.median():.0f} x {meta.h.median():.0f} px (w x h)")

In [ ]:
# CELL 09 · Table 1 and Figure 2: classes and class distribution
train_meta, test_meta = meta[meta.split == "train"], meta[meta.split == "validation"]
freq = pd.DataFrame({
    "train_images": [sum(c in l for l in train_meta.labels) for c in CLASSES],
    "test_images":  [sum(c in l for l in test_meta.labels) for c in CLASSES],
    "polygons":     [int((shapes_df.label == c).sum()) for c in CLASSES],
    "area_pct":     [100 * shapes_df.loc[(shapes_df.label == c) & (shapes_df.split == "train"), "area_frac"].sum()
                     / max(len(train_meta), 1) for c in CLASSES],
}, index=CLASSES)

# ---- Table 1 ----
group_rank = {"Concrete defect": 0, "General defect": 1, "Object": 2}
t1 = freq.assign(group=[CLASS_GROUP[c] for c in CLASSES], rank=[group_rank[CLASS_GROUP[c]] for c in CLASSES])
t1 = t1.sort_values(["rank", "train_images"], ascending=[True, False])
table1 = pd.DataFrame({
    "Label": t1.index, "Class": [CLASS_NAMES[c] for c in t1.index], "Group": t1.group,
    "Train images": [f"{v:,}" for v in t1.train_images], "Test images": [f"{v:,}" for v in t1.test_images],
    "Polygons": [f"{v:,}" for v in t1.polygons], "Area (%)": [f"{v:.2f}" for v in t1.area_pct]})
publish_table("classes", table1, raw=freq)

# ---- Figure 2 ----
order2 = freq.sort_values("train_images", ascending=True)      # largest class at the top
names2 = [CLASS_NAMES[c] for c in order2.index]

def draw_class_dist(layout):
    if layout == "horizontal":
        fig, axes = plt.subplots(1, 2, figsize=(COL2, 3.4), sharey=True)
    else:
        fig, axes = plt.subplots(2, 1, figsize=(COL1, 6.6))
    y = np.arange(len(order2))
    spec = [("train_images", "Training images containing the class", "(a) Images per class"),
            ("area_pct", "Share of image area (%)", "(b) Labelled area")]
    for ax, (col, xlabel, title) in zip(axes, spec):
        ax.barh(y, order2[col].values, color=PALETTE[0], height=0.72, edgecolor="white", linewidth=0.4)
        ax.set_yticks(y); ax.set_yticklabels(names2)
        ax.set_ylim(-0.6, len(y) - 0.4)
        ax.set_xlabel(xlabel); ax.set_title(title); ax.grid(axis="y", visible=False)
        ax.set_xlim(0, max(order2[col].max(), 1e-9) * 1.05)
    if layout == "horizontal":
        axes[1].tick_params(labelleft=False)
    return fig

publish_figure("class_dist", draw_class_dist)

In [ ]:
# CELL 10 · Check image sizes against annotations
# Image-annotation size consistency check on a sample (reads only JPEG headers via PIL)
sample = meta.sample(min(300, len(meta)), random_state=0)
mismatch = []
for r in sample.itertuples():
    with Image.open(os.path.join(cfg.data_root, "images", r.split, r.name)) as im:
        if im.size != (r.w, r.h):
            mismatch.append((r.name, im.size, (r.w, r.h)))
print(f"Size mismatches in {len(sample)} sampled images: {len(mismatch)}")
for m in mismatch[:5]: print("  ", m)

### 3.1 Looking at the annotations

Plotting a few images with their polygons is the fastest way to understand what "coarse polygon annotation" means in practice, and to spot systematic problems. Only the classes present in each image get a colour, in a fixed order; outlines are drawn on top so overlapping labels stay visible.

In [ ]:
# CELL 11 · Figure 3: annotated examples
def load_original(name, split):
    img = cv2.imread(os.path.join(cfg.data_root, "images", split, name), cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def annotated_image(name, split, colours, max_side=1000):
    img = load_original(name, split)
    h, w = img.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA) if scale < 1 else img
    ann = json.load(open(os.path.join(cfg.data_root, "annotations", split, name.replace(".jpg", ".json"))))
    overlay, outline = img.copy(), img.copy()
    thick = max(2, int(round(img.shape[1] / 300)))
    for sh in ann["shapes"]:
        if sh["label"] not in colours:
            continue
        rgb = tuple(int(colours[sh["label"]][k:k + 2], 16) for k in (1, 3, 5))
        pts = (np.asarray(sh["points"], np.float64) * scale).astype(np.int32)
        cv2.fillPoly(overlay, [pts], rgb)
        cv2.polylines(outline, [pts], True, rgb, thick, lineType=cv2.LINE_AA)
    blend = cv2.addWeighted(overlay, 0.35, img, 0.65, 0)
    edge = np.any(outline != img, axis=2)
    blend[edge] = outline[edge]
    return crop_to_aspect(blend)

# Pick six training images with 2-4 classes each, so that all shown classes fit one colour palette
cands = meta[meta.split == "train"].copy()
cands["k"] = cands.labels.apply(len)
cands = cands[(cands.k >= 2) & (cands.k <= 4)].sample(frac=1.0, random_state=3)
cands["near_4_3"] = (cands.w / cands.h - 4 / 3).abs() < 0.1          # prefer photos that need little cropping
cands = cands.sort_values("near_4_3", ascending=False, kind="stable")
chosen, union = [], []
for r in cands.itertuples():
    new = [c for c in r.labels if c in CLASS_INDEX and c not in union]
    if len(union) + len(new) <= len(PALETTE):
        chosen.append(r); union += new
    if len(chosen) == 6:
        break
union = sorted(union, key=CLASS_INDEX.get)
colours = {c: PALETTE[i] for i, c in enumerate(union)}
example_imgs = [annotated_image(r.name, r.split, colours) for r in chosen]
print("Examples:", [r.name for r in chosen])

example_imgs = example_imgs[:6] if len(example_imgs) >= 6 else example_imgs[:4] if len(example_imgs) >= 4 else example_imgs
n_ex = len(example_imgs)

def draw_examples(layout):
    grid = {6: ((2, 3), (3, 2)), 4: ((2, 2), (2, 2))}.get(n_ex, ((1, n_ex), (n_ex, 1)))
    (nr, nc), W = (grid[0], COL2) if layout == "horizontal" else (grid[1], COL1)
    cell_w = W / nc
    H = nr * (cell_w * 0.78 + 0.2) + (0.45 if layout == "horizontal" else 0.75)
    fig, axes = plt.subplots(nr, nc, figsize=(W, H), squeeze=False)
    for i, ax in enumerate(axes.flat):
        if i < len(example_imgs):
            image_axes(ax, example_imgs[i], title=f"({'abcdef'[i]})", title_loc="left", bold=True)
        else:
            ax.axis("off")
    handles = [Patch(facecolor=colours[c], edgecolor=colours[c], label=CLASS_NAMES[c]) for c in union]
    legend_below(fig, handles, [h.get_label() for h in handles])
    return fig

publish_figure("examples", draw_examples)

---
## 4. From polygons to cached training data

Loading a 4-megapixel JPEG and rasterising 19 polygon masks on every training step would make the GPU wait for the CPU. We do it **once**:

- Each image is resized to `img_size × img_size` with area interpolation and saved as a high-quality JPEG.
- Polygons are scaled to the target size and **rasterised directly at that size** with `cv2.fillPoly` using 4 bits of sub-pixel precision. This is faster than drawing at full resolution and resizing, and avoids blurring thin cracks.
- The 19 binary channels are bit-packed and saved compressed (a few kB per image).

Two details matter for correctness:

- **Multi-label masks.** Each class has its own channel, so overlapping labels (Spalling + Exposed Rebars + Rust on the same pixels) are preserved. Overlapping polygons of one class are merged, as in the official toolkit.
- **EXIF orientation is ignored** when reading images (`IMREAD_IGNORE_ORIENTATION`). The polygon coordinates refer to the stored pixel grid, which is how the official toolkit reads them with PIL.

Like the published baselines, images are resized to a square, which distorts the aspect ratio. Section 9.3 (resolution study) tests whether this costs accuracy on thin defects.

In [ ]:
# CELL 12 · Rasterise polygons into cached masks
SHIFT = 4  # sub-pixel precision bits for cv2.fillPoly

def rasterize(shapes, src_w, src_h, size):
    mask = np.zeros((C, size, size), np.uint8)
    sx, sy = size / src_w, size / src_h
    for s in shapes:
        c = CLASS_INDEX.get(s["label"])
        if c is None: continue
        pts = np.asarray(s["points"], np.float64)
        if s.get("shape_type") == "rectangle" and len(pts) == 2:
            (x0, y0), (x1, y1) = pts
            pts = np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]])
        if len(pts) < 3: continue
        pts = pts * [sx, sy]
        pts = np.round(pts * (1 << SHIFT)).astype(np.int32)
        cv2.fillPoly(mask[c], [pts], 1, lineType=cv2.LINE_8, shift=SHIFT)
    return mask

def cache_dir(size):
    return os.path.join(cfg.data_root, f"cache_{size}")

def _cache_one(args):
    split, name, size = args
    out_img = os.path.join(cache_dir(size), "images", name)
    out_msk = os.path.join(cache_dir(size), "masks", name.replace(".jpg", ".npz"))
    if os.path.exists(out_img) and (split not in ("train", "validation") or os.path.exists(out_msk)):
        return
    img = cv2.imread(os.path.join(cfg.data_root, "images", split, name), cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    h, w = img.shape[:2]
    cv2.imwrite(out_img, cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA), [cv2.IMWRITE_JPEG_QUALITY, 95])
    ann_path = os.path.join(cfg.data_root, "annotations", split, name.replace(".jpg", ".json"))
    if os.path.exists(ann_path):
        ann = json.load(open(ann_path))
        m = rasterize(ann["shapes"], ann.get("imageWidth", w), ann.get("imageHeight", h), size)
        np.savez_compressed(out_msk, m=np.packbits(m))

def build_cache(size, splits=("train", "validation", "testdev")):
    os.makedirs(os.path.join(cache_dir(size), "images"), exist_ok=True)
    os.makedirs(os.path.join(cache_dir(size), "masks"), exist_ok=True)
    jobs = [(s, os.path.basename(p), size) for s in splits
            for p in sorted(glob.glob(os.path.join(cfg.data_root, "images", s, "*.jpg")))]
    n_proc = max(1, (os.cpu_count() or 2))
    with get_context("fork").Pool(n_proc) as pool:
        list(tqdm(pool.imap_unordered(_cache_one, jobs, chunksize=8), total=len(jobs), desc=f"Caching {size}px"))

def load_mask(name, size):
    packed = np.load(os.path.join(cache_dir(size), "masks", name.replace(".jpg", ".npz")))["m"]
    return np.unpackbits(packed)[: C * size * size].reshape(C, size, size)

def load_image(name, size):
    img = cv2.imread(os.path.join(cache_dir(size), "images", name), cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

t0 = time.time()
build_cache(cfg.img_size)
print(f"Cache ready in {time.time() - t0:.0f}s")

# Sanity check: every cached mask channel must match the class labels present in its JSON
bad = 0
for r in meta.sample(min(200, len(meta)), random_state=1).itertuples():
    m = load_mask(r.name, cfg.img_size)
    present = {CLASSES[c] for c in range(C) if m[c].any()}
    if not present.issubset(set(r.labels)):
        bad += 1
print("Mask/label inconsistencies in sample:", bad)

---
## 5. Leakage audit and data splits

### 5.1 Why a leakage audit

The official splits were drawn from 20 image-similarity clusters, and the dataset paper does not say that photos of the same bridge are kept in one split. Inspection photos of one structure are often taken seconds apart. If near-identical photos sit in both training and test data, the test set partly measures memorisation and **every reported score is inflated**.

No bridge identifiers are published, so we approximate with a **perceptual hash (pHash)**: a 64-bit fingerprint that changes little under resizing and small exposure changes. Two images whose hashes differ in at most `phash_max_dist` bits are treated as near-duplicates. Look at the example pairs printed below and adjust the threshold if it flags clearly different scenes or misses obvious repeats.

### 5.2 How the splits are built

| Split used here | Source | Purpose |
|---|---|---|
| `train` | ~90% of official train | Model fitting |
| `val` | ~10% of official train | Early stopping, threshold tuning, temperature scaling |
| `test` | Official validation (975) | Final evaluation, touched once per model |
| `test_clean` | `test` minus images with a near-duplicate in official train | Leakage-robust re-evaluation |

The internal train/val split is **grouped by near-duplicate clusters**, so a scene never appears on both sides. Results are reported on both `test` and `test_clean`; a large gap between them is itself a finding.

In [ ]:
# CELL 13 · Near-duplicate (leakage) audit
def phash_int(name, size):
    with Image.open(os.path.join(cache_dir(size), "images", name)) as im:
        return int(str(imagehash.phash(im)), 16)

labelled = meta[["name", "split"]].reset_index(drop=True)
hashes = np.array([phash_int(n, cfg.img_size) for n in tqdm(labelled.name, desc="pHash")], dtype=np.uint64)

def popcount64(x):
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    return np.unpackbits(x.view(np.uint8).reshape(*x.shape, 8), axis=-1).sum(-1)

pairs = []
for i0 in range(0, len(hashes), 512):
    d = popcount64(hashes[i0:i0 + 512, None] ^ hashes[None, :])
    ii, jj = np.nonzero(d <= cfg.phash_max_dist)
    for a, b in zip(ii + i0, jj):
        if a < b: pairs.append((a, b, int(d[a - i0, b])))
pairs = pd.DataFrame(pairs, columns=["i", "j", "dist"])
pairs["split_i"] = labelled.split.values[pairs.i] if len(pairs) else []
pairs["split_j"] = labelled.split.values[pairs.j] if len(pairs) else []
cross = pairs[pairs.split_i != pairs.split_j] if len(pairs) else pairs
print(f"Near-duplicate pairs (Hamming <= {cfg.phash_max_dist}): {len(pairs)} total, {len(cross)} across official train/validation")

# Union-find over near-duplicate pairs -> groups used for grouped splitting
parent = list(range(len(labelled)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, b in zip(pairs.i, pairs.j):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb
labelled["group"] = [find(i) for i in range(len(labelled))]
print("Images in a duplicate group of size > 1:", int((labelled.groupby("group").name.transform("size") > 1).sum()))

In [ ]:
# CELL 14 · Figure S1: near-duplicate pairs
if len(cross):
    show = cross.sort_values("dist").head(4)
    leak_pairs = []
    for r in show.itertuples():
        a, b = (r.i, r.j) if labelled.split[r.i] == "train" else (r.j, r.i)
        leak_pairs.append((crop_to_aspect(thumbnail(load_original(labelled.name[a], labelled.split[a]))),
                           crop_to_aspect(thumbnail(load_original(labelled.name[b], labelled.split[b]))), r.dist))

    def draw_leakage(layout):
        n = len(leak_pairs)
        if layout == "horizontal":   # each pair is a column: training image above, test image below
            W = min(COL2, n * 1.9 + 0.35)
            cw = (W - 0.35) / n
            fig, axes = plt.subplots(2, n, figsize=(W, 2 * cw * 0.8 + 0.45), squeeze=False)
            for j, (tr_img, te_img, d) in enumerate(leak_pairs):
                image_axes(axes[0, j], tr_img, title=f"({'abcd'[j]}) d = {d}", title_loc="left", bold=True,
                           ylabel="Training" if j == 0 else None)
                image_axes(axes[1, j], te_img, ylabel="Test" if j == 0 else None)
        else:                        # each pair is a row: training image left, test image right
            fig, axes = plt.subplots(n, 2, figsize=(COL1, n * 1.35 * 0.8 + 0.4), squeeze=False)
            for i, (tr_img, te_img, d) in enumerate(leak_pairs):
                image_axes(axes[i, 0], tr_img, title="Training" if i == 0 else None, ylabel=f"({'abcd'[i]}) d = {d}")
                image_axes(axes[i, 1], te_img, title="Test" if i == 0 else None)
        return fig

    publish_figure("leakage", draw_leakage)
else:
    print("No cross-split near-duplicates found at this threshold; Figure S1 is not produced.")

In [ ]:
# CELL 15 · Create data splits and Table S1
split_path = os.path.join(cfg.project_dir, "splits", "splits.csv")
if os.path.exists(split_path):
    splits = pd.read_csv(split_path)
    print("Loaded existing splits from Drive (kept fixed across sessions).")
else:
    tr = labelled[labelled.split == "train"].reset_index(drop=True)
    gss = GroupShuffleSplit(n_splits=1, test_size=cfg.val_fraction, random_state=42)
    tr_idx, va_idx = next(gss.split(tr, groups=tr.group))
    test = labelled[labelled.split == "validation"].copy()
    train_groups = set(tr.group)
    test["clean"] = ~test.group.isin(train_groups)
    splits = pd.concat([
        pd.DataFrame({"name": tr.name.iloc[tr_idx], "subset": "train", "clean": True}),
        pd.DataFrame({"name": tr.name.iloc[va_idx], "subset": "val", "clean": True}),
        pd.DataFrame({"name": test.name, "subset": "test", "clean": test.clean.values}),
    ], ignore_index=True)
    splits.to_csv(split_path, index=False)
    print("Created and saved new splits.")

SPLIT = {s: splits[splits.subset == s].name.tolist() for s in ["train", "val", "test"]}
SPLIT["test_clean"] = splits[(splits.subset == "test") & splits.clean].name.tolist()
for k, v in SPLIT.items():
    print(f"{k:10s}: {len(v):5d} images")

table_s1 = pd.DataFrame([
    ["Training", f"{len(SPLIT['train']):,}", "Official training split (about 90%)", "Model fitting"],
    ["Validation", f"{len(SPLIT['val']):,}", "Official training split (about 10%), grouped by near-duplicates",
     "Early stopping, thresholds, temperature scaling"],
    ["Test", f"{len(SPLIT['test']):,}", "Official validation split", "Final evaluation"],
    ["Test, leakage-free", f"{len(SPLIT['test_clean']):,}", "Test images without a near-duplicate in the training data",
     "Leakage-robust evaluation"],
], columns=["Subset", "Images", "Source", "Purpose"])
publish_table("splits", table_s1, n_pairs=len(cross), d=cfg.phash_max_dist)

---
## 6. Dataset, augmentation and data loaders

**Augmentation** must change image and mask identically, which albumentations handles. We use mild, physically plausible transforms: horizontal flips, small affine changes (scale ±20%, rotation ±10°), and brightness/contrast/colour jitter that mimics varying daylight and camera settings. Vertical flips are excluded from training because gravity-driven defects (efflorescence, wet spots, rust streaks) have a natural up–down direction.

**Normalisation** uses ImageNet statistics, which all three pretrained backbones expect.

The dataset can also apply a **corruption** (blur, noise, low light, JPEG) for the robustness study in Section 13.

In [ ]:
# CELL 16 · Data loaders and augmentation preview
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def corrupt(img, kind, severity):
    """img: uint8 RGB. severity in {1,2,3}."""
    if kind == "blur":
        return cv2.GaussianBlur(img, (0, 0), sigmaX=[1.0, 2.0, 4.0][severity - 1])
    if kind == "noise":
        rng = np.random.default_rng(severity)
        return np.clip(img + rng.normal(0, [8, 16, 32][severity - 1], img.shape), 0, 255).astype(np.uint8)
    if kind == "dark":
        return np.clip(img.astype(np.float32) * [0.7, 0.5, 0.3][severity - 1], 0, 255).astype(np.uint8)
    if kind == "jpeg":
        ok, enc = cv2.imencode(".jpg", img[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, [30, 15, 8][severity - 1]])
        return cv2.imdecode(enc, cv2.IMREAD_COLOR)[..., ::-1].copy()
    raise ValueError(kind)

def train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.8, 1.2), rotate=(-10, 10), translate_percent=(-0.05, 0.05), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    ])

class DaclDataset(Dataset):
    def __init__(self, names, size, train=False, with_mask=True, corruption=None):
        self.names, self.size, self.with_mask, self.corruption = list(names), size, with_mask, corruption
        self.tf = train_transforms() if train else None

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        name = self.names[i]
        img = load_image(name, self.size)
        if self.corruption is not None:
            img = corrupt(img, *self.corruption)
        mask = load_mask(name, self.size) if self.with_mask else np.zeros((C, self.size, self.size), np.uint8)
        if self.tf is not None:
            out = self.tf(image=img, mask=np.ascontiguousarray(mask.transpose(1, 2, 0)))
            img, mask = out["image"], out["mask"].transpose(2, 0, 1)
        x = torch.from_numpy(((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1).copy())
        y = torch.from_numpy(np.ascontiguousarray(mask)).float()
        return x, y, i

def make_loader(names, train=False, size=None, batch_size=None, corruption=None, shuffle=None):
    import gc
    gc.collect()  # free finished loaders first, so new worker processes do not inherit them
    ds = DaclDataset(names, size or cfg.img_size, train=train, corruption=corruption)
    return DataLoader(ds, batch_size=batch_size or cfg.batch_size, shuffle=train if shuffle is None else shuffle,
                      num_workers=cfg.num_workers, pin_memory=DEVICE.type == "cuda", drop_last=train,
                      persistent_workers=train and cfg.num_workers > 0)  # one-pass evaluation needs no persistent workers

def denorm(x):
    """(3,H,W) normalised tensor -> uint8 RGB numpy image."""
    return (np.clip(x.cpu().numpy().transpose(1, 2, 0) * STD + MEAN, 0, 1) * 255).astype(np.uint8)

# Quick visual check of one augmented batch
xb, yb, _ = next(iter(make_loader(SPLIT["train"], train=True, batch_size=4)))
print("batch:", tuple(xb.shape), tuple(yb.shape), "| positive pixel share:", f"{yb.mean().item():.4f}")
fig, axes = plt.subplots(1, 4, figsize=(COL2, 2.2))
for ax, x, y in zip(axes, xb, yb):
    img = denorm(x)
    edge = np.zeros(img.shape[:2], bool)
    for c in range(C):
        m = y[c].numpy().astype(np.uint8)
        edge |= (cv2.morphologyEx(m, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0)
    img[edge] = (235, 104, 52)
    ax.imshow(img); ax.axis("off")
    present = [CLASS_NAMES[CLASSES[c]] for c in range(C) if y[c].any()]
    title = ", ".join(present[:2]) + (f" + {len(present) - 2} more" if len(present) > 2 else "")
    ax.set_title(title, fontsize=6, color=INK2, fontweight="normal", loc="center")
fig.suptitle("Diagnostic only (not saved): augmented training samples with mask outlines", fontsize=8)
plt.show()

---
## 7. Models

All models share one interface: `forward(x)` returns a dict with

- `logits` of shape `(B, 19, H, W)`: one independent **sigmoid** channel per class. Softmax is wrong here because one pixel can carry several labels.
- `aux` of shape `(B, 19)`: an image-level "is this class present?" head, as in the dacl10k baseline, which adds a cheap global signal during training.

Each model also exposes `cam_layer`, the feature map used for Grad-CAM in Section 12, and `param_groups()`, which gives pretrained encoders a smaller learning rate than freshly initialised decoders.

| Key | Architecture | Trainable part | Role in the study |
|---|---|---|---|
| `fpn_b4` | FPN + EfficientNet-B4 (ImageNet) | All | Replicates the best published dacl10k baseline |
| `segformer` | SegFormer MiT-B2 | All | Transformer baseline used by the challenge organisers |
| `dino_lora` | DINOv2 ViT-B/14 + LoRA + light decoder | LoRA adapters + decoder | Main proposed model |
| `dino_full` | Same, fully fine-tuned | All | Control: separates the effect of the backbone from the effect of LoRA |

**The DINOv2 decoder.** DINOv2 cuts the image into 14 × 14 patches, so a 512 px input becomes a 36 × 36 token grid after rounding. We resize inputs to the nearest multiple of 14 (518 px → 37 × 37 tokens), take tokens from four depths of the network (shallow layers keep edges, deep layers carry semantics), reshape them into feature maps, fuse them with a 1 × 1 convolution, then upsample in two learned ×2 stages before a final bilinear upsample. Thin cracks are the hardest case for this coarse grid, which is why Section 9.3 tests a higher resolution.

**LoRA** adds small trainable low-rank matrices to the attention query and value projections and freezes the 86 M backbone weights. With rank 16 on the query and value projections of all 12 layers it trains about 0.6 M backbone parameters instead of 86 M, which reduces overfitting risk on ~6,000 images and keeps memory low. Its dropout also gives MC dropout (Section 10) a source of randomness inside the backbone.

In [ ]:
# CELL 17 · Models and Table 2
def is_dropout(m):
    return isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout1d, nn.Dropout3d))

class FPNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = smp.FPN(encoder_name=cfg.fpn_encoder, encoder_weights="imagenet" if cfg.pretrained else None,
                           classes=C, decoder_dropout=0.2, aux_params=dict(classes=C, dropout=0.2))

    @property
    def cam_layer(self):  # property, not a registered submodule, so weights are not stored twice
        return self.net.decoder

    def forward(self, x):
        logits, aux = self.net(x)
        return {"logits": logits, "aux": aux}

    def param_groups(self):
        enc = [p for n, p in self.named_parameters() if n.startswith("net.encoder.") and p.requires_grad]
        rest = [p for n, p in self.named_parameters() if not n.startswith("net.encoder.") and p.requires_grad]
        return [{"params": enc, "lr": cfg.lr_backbone}, {"params": rest, "lr": cfg.lr_head}]

class SegFormerModel(nn.Module):
    def __init__(self):
        super().__init__()
        from transformers import SegformerForSemanticSegmentation, SegformerConfig
        if cfg.pretrained:
            self.net = SegformerForSemanticSegmentation.from_pretrained(cfg.segformer_name, num_labels=C,
                                                                        ignore_mismatched_sizes=True)
        else:  # tiny random model for the smoke test
            self.net = SegformerForSemanticSegmentation(SegformerConfig(
                num_labels=C, depths=[1, 1, 1, 1], hidden_sizes=[16, 32, 64, 128],
                num_attention_heads=[1, 1, 2, 4], decoder_hidden_size=64))
        hid = self.net.config.hidden_sizes[-1]
        self.aux_head = nn.Sequential(nn.Dropout(0.2), nn.Linear(hid, C))

    @property
    def cam_layer(self):
        return self.net.decode_head.linear_fuse

    def forward(self, x):
        out = self.net(pixel_values=x, output_hidden_states=True)
        logits = F.interpolate(out.logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        last = out.hidden_states[-1]
        pooled = last.mean(dim=(2, 3)) if last.ndim == 4 else last.mean(dim=1)
        return {"logits": logits, "aux": self.aux_head(pooled)}

    def param_groups(self):
        enc = [p for n, p in self.named_parameters() if n.startswith("net.segformer.") and p.requires_grad]
        rest = [p for n, p in self.named_parameters() if not n.startswith("net.segformer.") and p.requires_grad]
        return [{"params": enc, "lr": cfg.lr_backbone}, {"params": rest, "lr": cfg.lr_head}]

class DinoSegModel(nn.Module):
    def __init__(self, lora=True):
        super().__init__()
        from transformers import Dinov2Model, Dinov2Config
        if cfg.pretrained:
            self.backbone = Dinov2Model.from_pretrained(cfg.dino_name)
        else:
            self.backbone = Dinov2Model(Dinov2Config(hidden_size=64, num_hidden_layers=4, num_attention_heads=4,
                                                     intermediate_size=128, image_size=126))
        bc = self.backbone.config
        self.patch = bc.patch_size
        self.n_reg = int(getattr(bc, "num_register_tokens", 0) or 0)
        L, D = bc.num_hidden_layers, bc.hidden_size
        self.layer_ids = sorted({max(1, L // 4), max(1, L // 2), max(1, 3 * L // 4), L})
        self.input_size = max(self.patch, int(round(cfg.img_size / self.patch)) * self.patch)
        if cfg.grad_checkpointing:
            self.backbone.gradient_checkpointing_enable()
            if lora:  # frozen inputs would otherwise cut the gradient path through checkpoints
                self.backbone.enable_input_require_grads()
        self.lora = lora
        if lora:
            from peft import LoraConfig, inject_adapter_in_model
            targets = sorted({n.split(".")[-1] for n, m in self.backbone.named_modules()
                              if isinstance(m, nn.Linear) and n.split(".")[-1] in ("query", "value", "q_proj", "v_proj")})
            assert targets, "Could not find attention projections for LoRA"
            lcfg = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
                              target_modules=targets)
            self.backbone = inject_adapter_in_model(lcfg, self.backbone)
            for n, p in self.backbone.named_parameters():
                p.requires_grad = "lora_" in n
        k = len(self.layer_ids)
        self.proj = nn.Sequential(nn.Conv2d(k * D, 256, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up1 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.drop = nn.Dropout2d(0.1)
        self.cls = nn.Conv2d(64, C, 1)
        self.aux_head = nn.Sequential(nn.Dropout(0.2), nn.Linear(D, C))

    @property
    def cam_layer(self):
        return self.up2

    def forward(self, x):
        H, W = x.shape[-2:]
        s = self.input_size
        xi = F.interpolate(x, size=(s, s), mode="bilinear", align_corners=False) if (H, W) != (s, s) else x
        out = self.backbone(pixel_values=xi, output_hidden_states=True)
        g = s // self.patch
        feats = []
        for i in self.layer_ids:
            t = out.hidden_states[i][:, 1 + self.n_reg:, :]
            feats.append(t.transpose(1, 2).reshape(t.shape[0], -1, g, g))
        f = self.proj(torch.cat(feats, 1))
        f = self.up1(F.interpolate(f, scale_factor=2, mode="bilinear", align_corners=False))
        f = self.up2(F.interpolate(f, scale_factor=2, mode="bilinear", align_corners=False))
        logits = F.interpolate(self.cls(self.drop(f)), size=(H, W), mode="bilinear", align_corners=False)
        return {"logits": logits, "aux": self.aux_head(out.last_hidden_state[:, 0])}

    def param_groups(self):
        bb = [p for n, p in self.named_parameters() if n.startswith("backbone.") and p.requires_grad]
        rest = [p for n, p in self.named_parameters() if not n.startswith("backbone.") and p.requires_grad]
        # LoRA adapters start at zero and need the larger learning rate; full fine-tuning needs the small one
        return [{"params": bb, "lr": cfg.lr_head if self.lora else cfg.lr_backbone},
                {"params": rest, "lr": cfg.lr_head}]

MODEL_BUILDERS = {
    "fpn_b4": FPNModel,
    "segformer": SegFormerModel,
    "dino_lora": lambda: DinoSegModel(lora=True),
    "dino_full": lambda: DinoSegModel(lora=False),
}

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train

# Build each model once on a dummy batch: checks shapes and reports parameter counts
param_table = []
for key, build in MODEL_BUILDERS.items():
    m = build().to(DEVICE).eval()
    with torch.no_grad():
        o = m(torch.zeros(1, 3, cfg.img_size, cfg.img_size, device=DEVICE))
    tot, tr = count_params(m)
    param_table.append({"model": key, "total_params_M": tot / 1e6, "trainable_params_M": tr / 1e6,
                        "logits": tuple(o["logits"].shape), "aux": tuple(o["aux"].shape)})
    del m
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
param_table = pd.DataFrame(param_table).set_index("model")
print(param_table[["logits", "aux"]].to_string())

PRETRAINED = {"fpn_b4": "ImageNet-1k (supervised)", "segformer": "ImageNet-1k (supervised)",
              "dino_lora": "LVD-142M (self-supervised)", "dino_full": "LVD-142M (self-supervised)"}
TRAINED_PART = {"fpn_b4": "All weights", "segformer": "All weights",
                "dino_lora": "LoRA adapters and decoder", "dino_full": "All weights"}
table2 = pd.DataFrame({
    "Model": [MODEL_NAMES[k] for k in MODEL_KEYS],
    "Pretrained weights": [PRETRAINED[k] if cfg.pretrained else "None (dry run)" for k in MODEL_KEYS],
    "Trained part": [TRAINED_PART[k] for k in MODEL_KEYS],
    "Parameters (M)": [f"{param_table.loc[k, 'total_params_M']:.2f}" for k in MODEL_KEYS],
    "Trainable (M)": [f"{param_table.loc[k, 'trainable_params_M']:.2f}" for k in MODEL_KEYS]})
publish_table("models", table2, raw=param_table)

---
## 8. Loss function and training loop

### 8.1 Loss

dacl10k is extremely imbalanced: background covers about 73% of pixels, and Exposed Rebars, Restformwork and Joint Tape under 0.2% each. Plain binary cross-entropy would let the model ignore rare classes. We therefore combine three terms:

$$\mathcal{L} = \underbrace{\frac{1}{|\mathcal{P}|}\sum_{c\in\mathcal{P}} \mathcal{L}^{(c)}_{\text{Dice}}}_{\text{overlap, per class}} \;+\; \lambda_f\,\mathcal{L}_{\text{focal}} \;+\; \lambda_a\,\mathcal{L}_{\text{aux}}$$

- **Soft Dice**, computed per class over the batch, directly optimises overlap and is insensitive to class frequency. It is averaged only over classes present in the batch ($\mathcal{P}$), so an absent class does not produce an undefined score.
- **Focal BCE** (γ = 2) penalises false positives everywhere, including classes absent from the batch, while down-weighting the many easy background pixels.
- **Auxiliary BCE** on image-level presence labels, with weight 0.1 as in the dacl10k baseline.

### 8.2 Training loop

- **AdamW** with separate learning rates for encoder and decoder, **linear warm-up then cosine decay**, gradient clipping and mixed precision (bfloat16 where the GPU supports it).
- After every epoch the model is scored on the internal `val` split (mIoU at threshold 0.5). The **best epoch is kept** and training stops after `patience` epochs without improvement.
- **Resumable:** the full training state is written to Drive each epoch. If Colab disconnects, re-run the cell and training continues where it stopped. Finished runs are skipped.
- For LoRA models only trainable weights and buffers are saved, which keeps checkpoints small; the frozen backbone is reloaded from the Hugging Face hub.

In [ ]:
# CELL 18 · Loss function
from torchvision.ops import sigmoid_focal_loss

def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits.float())
    dims = (0, 2, 3)
    inter = (p * target).sum(dims)
    denom = p.sum(dims) + target.sum(dims)
    dice = 1.0 - (2 * inter + eps) / (denom + eps)
    present = target.sum(dims) > 0
    return dice[present].mean() if present.any() else dice.sum() * 0.0

def total_loss(out, y):
    logits = out["logits"].float()
    l_dice = dice_loss(logits, y)
    l_focal = sigmoid_focal_loss(logits, y, alpha=-1, gamma=2.0, reduction="mean")
    l_aux = F.binary_cross_entropy_with_logits(out["aux"].float(), (y.amax(dim=(2, 3)) > 0).float())
    loss = l_dice + cfg.focal_weight * l_focal + cfg.aux_weight * l_aux
    return loss, {"dice": l_dice.item(), "focal": l_focal.item(), "aux": l_aux.item()}

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=cfg.amp)

In [ ]:
# CELL 19 · Training loop
@torch.no_grad()
def quick_val_miou(model, loader, thr=0.5):
    """Dataset-level IoU per class at a fixed threshold (used for model selection during training)."""
    model.eval()
    inter = torch.zeros(C, device=DEVICE); union = torch.zeros(C, device=DEVICE)
    for x, y, _ in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with autocast():
            p = torch.sigmoid(model(x)["logits"].float())
        pred = p > thr
        yb = y > 0.5
        inter += (pred & yb).sum(dim=(0, 2, 3)).float()
        union += (pred | yb).sum(dim=(0, 2, 3)).float()
    valid = union > 0
    return (inter[valid] / union[valid]).mean().item()

def saveable_state(model):
    """All trainable parameters + all buffers (BatchNorm stats). Frozen pretrained weights are skipped."""
    keep = {n for n, p in model.named_parameters() if p.requires_grad} | {n for n, _ in model.named_buffers()}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in keep}

def load_saveable_state(model, state):
    missing, unexpected = model.load_state_dict(state, strict=False)
    frozen = {n for n, p in model.named_parameters() if not p.requires_grad}
    bad_missing = [k for k in missing if k not in frozen]
    assert not bad_missing and not unexpected, f"State mismatch: missing={bad_missing[:5]} unexpected={unexpected[:5]}"

def run_dir(model_key, seed, size=None):
    return os.path.join(cfg.project_dir, "runs", f"{model_key}_s{seed}_{size or cfg.img_size}px")

def run_finished(model_key, seed, size=None):
    """True only when training completed (DONE marker); an interrupted run also has a best.pt, but is not finished."""
    return os.path.exists(os.path.join(run_dir(model_key, seed, size), "DONE"))

def train_model(model_key, seed, size=None, epochs=None):
    size = size or cfg.img_size
    epochs = epochs or cfg.epochs
    rd = run_dir(model_key, seed, size); os.makedirs(rd, exist_ok=True)
    if os.path.exists(os.path.join(rd, "DONE")):
        print(f"[{model_key} seed {seed} {size}px] already finished, skipping."); return rd
    seed_everything(seed)
    model = MODEL_BUILDERS[model_key]().to(DEVICE)
    bs = cfg.batch_size if size == cfg.img_size else max(1, cfg.batch_size // 4)
    accum = cfg.accum_steps * (cfg.batch_size // bs)
    tr_loader = make_loader(SPLIT["train"], train=True, size=size, batch_size=bs)
    va_loader = make_loader(SPLIT["val"], size=size, batch_size=bs)
    opt = torch.optim.AdamW(model.param_groups(), weight_decay=cfg.weight_decay)
    steps_per_epoch = max(1, len(tr_loader) // accum)
    total_steps, warm = epochs * steps_per_epoch, cfg.warmup_epochs * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, warm) if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))
    scaler = torch.amp.GradScaler(enabled=cfg.amp and AMP_DTYPE == torch.float16)
    start_epoch, best, bad_epochs, history = 0, -1.0, 0, []
    last_path = os.path.join(rd, "last.pt")
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        load_saveable_state(model, ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        start_epoch, best, bad_epochs, history = ck["epoch"] + 1, ck["best"], ck["bad_epochs"], ck["history"]
        print(f"Resumed from epoch {start_epoch}")
    for epoch in range(start_epoch, epochs):
        model.train(); t0 = time.time(); running = []
        opt.zero_grad(set_to_none=True)
        for step, (x, y, _) in enumerate(tqdm(tr_loader, desc=f"{model_key} s{seed} ep{epoch + 1}/{epochs}", leave=False)):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                out = model(x)
            loss, parts = total_loss(out, y)
            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            running.append(loss.item())
        val_miou = quick_val_miou(model, va_loader)
        history.append({"epoch": epoch + 1, "train_loss": float(np.mean(running)), "val_mIoU@0.5": val_miou,
                        "minutes": (time.time() - t0) / 60})
        improved = val_miou > best
        if improved:
            best, bad_epochs = val_miou, 0
            torch.save(saveable_state(model), os.path.join(rd, "best.pt"))
        else:
            bad_epochs += 1
        torch.save({"model": saveable_state(model), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch, "best": best, "bad_epochs": bad_epochs,
                    "history": history}, last_path)
        pd.DataFrame(history).to_csv(os.path.join(rd, "history.csv"), index=False)
        print(f"  epoch {epoch + 1:2d} | loss {np.mean(running):.4f} | val mIoU {val_miou:.4f}"
              f"{' *' if improved else ''} | {history[-1]['minutes']:.1f} min")
        if bad_epochs >= cfg.patience:
            if epoch + 1 < epochs:          # only a real early stop when epochs were actually skipped
                print(f"  early stopping after epoch {epoch + 1} (best epoch {epoch + 1 - bad_epochs})")
            break
    open(os.path.join(rd, "DONE"), "w").write(json.dumps({"best_val_mIoU@0.5": best}))
    os.remove(last_path)  # optimizer state no longer needed
    del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    return rd

def load_trained(model_key, seed, size=None):
    model = MODEL_BUILDERS[model_key]().to(DEVICE)
    load_saveable_state(model, torch.load(os.path.join(run_dir(model_key, seed, size), "best.pt"),
                                          map_location="cpu", weights_only=False))
    return model.eval()

---
## 9. Experiment E1: accuracy of the four models (RQ1)

### 9.1 Evaluation machinery

Every later experiment reuses one evaluation pass, `run_eval`, which streams predictions over a split once and records:

1. **Per-image intersection and union for every class at every threshold** in a grid (0.10 to 0.90). This lets us tune one threshold per class on `val` and then score `test` *without re-running the model*, and it makes the image-level bootstrap in 9.4 possible.
2. Optionally, **a fixed random sample of pixels per image** (same pixel positions for every method, seeded by the image name) with predicted probability, uncertainty and label. Only the calibration and error-targeting analyses (Sections 10–11) need these, so they are switched on only there to save memory.
3. A **class-confusion matrix**: for ground-truth class *a*, the share of its pixels predicted as class *b*.
4. Mean uncertainty per image, for the robustness study.

Results are cached as `.npz` files on Drive, so re-running an analysis cell is instant.

**Metric definition.** IoU for class *c* is computed at dataset level, $\text{IoU}_c = \sum_i I_{i,c} / \sum_i U_{i,c}$, and mIoU averages over classes that occur in the split. Other papers sometimes average per image or per batch; the numbers are not directly comparable, so state this definition in the paper.

**Uncertainty decomposition.** For each pixel and class, with $S$ stochastic predictions $p_s$ and their mean $\bar p$, binary entropy $H$ gives

$$\underbrace{H(\bar p)}_{\text{total}} = \underbrace{\tfrac{1}{S}\textstyle\sum_s H(p_s)}_{\text{aleatoric (data)}} + \underbrace{\Big[H(\bar p) - \tfrac{1}{S}\textstyle\sum_s H(p_s)\Big]}_{\text{epistemic (model), mutual information}}$$

A deterministic model has $S = 1$, so its epistemic part is zero by construction.

In [ ]:
# CELL 20 · Evaluation machinery
import zlib
THR_GRID = torch.tensor(cfg.thr_grid, dtype=torch.float32)
EPS = 1e-6

def bin_entropy(p):
    p = p.clamp(EPS, 1 - EPS)
    return -(p * p.log() + (1 - p) * (1 - p).log())

def enable_mc_dropout(model):
    model.eval()
    for m in model.modules():
        if is_dropout(m):
            m.train()

def _forward_probs(model, x, input_size=None, scale=1.0, hflip=False):
    H, W = x.shape[-2:]
    s = int(round((input_size or H) * scale / 32) * 32) if (input_size or scale != 1.0) else H
    xi = F.interpolate(x, size=(s, s), mode="bilinear", align_corners=False, antialias=True) if s != H else x
    if hflip: xi = xi.flip(-1)
    with autocast():
        logits = model(xi)["logits"].float()
    if hflip: logits = logits.flip(-1)
    if logits.shape[-2:] != (H, W):
        logits = F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)
    return torch.sigmoid(logits)

@torch.no_grad()
def predict(models, x, method="single", T=None, input_size=None):
    """Returns mean probability and total/aleatoric/epistemic uncertainty, each (B, C, H, W)."""
    passes = []
    if method == "single":
        models[0].eval(); passes = [lambda: _forward_probs(models[0], x, input_size)]
    elif method == "mc":
        enable_mc_dropout(models[0]); passes = [lambda: _forward_probs(models[0], x, input_size)] * (T or cfg.mc_samples)
    elif method == "tta":
        models[0].eval()
        passes = [(lambda s=s, f=f: _forward_probs(models[0], x, input_size, scale=s, hflip=f))
                  for s in cfg.tta_scales for f in (False, True)]
    elif method == "ensemble":
        for m in models: m.eval()
        passes = [(lambda m=m: _forward_probs(m, x, input_size)) for m in models]
    else:
        raise ValueError(method)
    sum_p, sum_h = None, None
    for fn in passes:
        p = fn()
        h = bin_entropy(p)
        sum_p = p if sum_p is None else sum_p + p
        sum_h = h if sum_h is None else sum_h + h
    if method == "mc": models[0].eval()
    n = len(passes)
    p_mean = sum_p / n
    total = bin_entropy(p_mean)
    alea = sum_h / n
    return p_mean, total, alea, (total - alea).clamp_min(0)

def pixel_index(name, hw, k):
    rng = np.random.default_rng(zlib.crc32(name.encode()))
    return torch.from_numpy(rng.choice(hw, size=min(k, hw), replace=False))

def run_eval(models, names, method, tag, size=None, input_size=None, corruption=None, T=None,
             conf_thr=None, samples=False, overwrite=False):
    """One streaming pass over `names`. Results cached in PROJECT_DIR/results/<tag>.npz.
    samples=True also stores the per-pixel sample (needed only for calibration / error-targeting)."""
    path = os.path.join(cfg.project_dir, "results", f"{tag}.npz")
    if os.path.exists(path) and not overwrite:
        return dict(np.load(path, allow_pickle=True))
    size = size or cfg.img_size
    loader = make_loader(names, size=size, batch_size=max(1, cfg.batch_size // 2), corruption=corruption)
    G = THR_GRID.to(DEVICE).view(-1, 1, 1, 1, 1)
    conf_thr = torch.full((C,), 0.5) if conf_thr is None else torch.as_tensor(conf_thr, dtype=torch.float32)
    conf_thr = conf_thr.to(DEVICE).view(1, C, 1, 1)
    I_all, U_all, S_p, S_tot, S_alea, S_epi, S_y, unc_img = [], [], [], [], [], [], [], []
    conf = torch.zeros(C, C, device=DEVICE, dtype=torch.float64); gt_pix = torch.zeros(C, device=DEVICE, dtype=torch.float64)
    for x, y, idx in tqdm(loader, desc=tag, leave=False):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True) > 0.5
        p, tot, alea, epi = predict(models, x, method, T=T, input_size=input_size)
        I, U = [], []
        for g in G:  # loop over thresholds keeps memory bounded
            pred = p > g[0]
            I.append((pred & y).sum(dim=(2, 3))); U.append((pred | y).sum(dim=(2, 3)))
        I_all.append(torch.stack(I, 1).cpu().numpy().astype(np.int32))
        U_all.append(torch.stack(U, 1).cpu().numpy().astype(np.int32))
        B, _, H, W = p.shape
        if samples:
            pidx = torch.stack([pixel_index(loader.dataset.names[i], H * W, cfg.pixels_per_image) for i in idx.tolist()]).to(DEVICE)
            gather = lambda t: torch.gather(t.reshape(B, C, -1), 2, pidx[:, None, :].expand(B, C, -1)).cpu().numpy().astype(np.float16)
            S_p.append(gather(p)); S_tot.append(gather(tot)); S_alea.append(gather(alea)); S_epi.append(gather(epi))
            S_y.append(gather(y.float()).astype(bool))
        gt = y.reshape(B, C, -1).double(); pr = (p > conf_thr).reshape(B, C, -1).double()
        conf += torch.einsum("bcn,bdn->cd", gt, pr); gt_pix += gt.sum(dim=(0, 2))
        unc_img.append(np.stack([tot.mean(dim=(1, 2, 3)).cpu().numpy(), epi.mean(dim=(1, 2, 3)).cpu().numpy()], 1))
    res = {"names": np.array(loader.dataset.names), "I": np.concatenate(I_all), "U": np.concatenate(U_all),
           "unc_img": np.concatenate(unc_img),
           "conf": (conf / gt_pix.clamp_min(1)[:, None]).cpu().numpy(), "gt_pix": gt_pix.cpu().numpy()}
    if samples:
        res.update({"p": np.concatenate(S_p), "total": np.concatenate(S_tot), "alea": np.concatenate(S_alea),
                    "epi": np.concatenate(S_epi), "y": np.concatenate(S_y)})
    np.savez_compressed(path, **res)
    return res

def tune_thresholds(ev):
    """Index into THR_GRID per class maximising dataset-level IoU."""
    I, U = ev["I"].sum(0).astype(np.float64), ev["U"].sum(0).astype(np.float64)   # (T, C)
    iou = np.where(U > 0, I / np.maximum(U, 1), 0)
    best = iou.argmax(0)
    best[U.sum(0) == 0] = int(np.argmin(np.abs(np.array(cfg.thr_grid) - 0.5)))
    return best

def select(ev, thr_idx, subset=None):
    """Per-image I and U (N, C) at the chosen per-class thresholds, optionally for a subset of image names."""
    I = np.take_along_axis(ev["I"], thr_idx[None, None, :], 1)[:, 0]
    U = np.take_along_axis(ev["U"], thr_idx[None, None, :], 1)[:, 0]
    if subset is not None:
        keep = np.isin(ev["names"], list(subset)); I, U = I[keep], U[keep]
    return I.astype(np.float64), U.astype(np.float64)

def class_iou(I, U):
    Us = U.sum(0)
    return np.where(Us > 0, I.sum(0) / np.maximum(Us, 1), np.nan)

def miou(I, U):
    return float(np.nanmean(class_iou(I, U)))

### 9.2 Train the four models

This is the expensive cell. It loops over models and seeds and **skips any run that already finished**, so you can run it across several Colab sessions.

- **One seed:** `E1_SEEDS = [0]` trains each of the four models once, for a quicker first look.
- **Three seeds (as in the paper):** `E1_SEEDS = [0, 1, 2]`; if seed 0 has already finished, only the new seeds train. Three seeds give seed standard deviations and enable the deep ensemble.

Watch the printed minutes per epoch in the first run and multiply by `epochs × runs` to plan your GPU time.

In [ ]:
# CELL 21 · Train the models (E1)
E1_MODELS = list(MODEL_KEYS)
E1_SEEDS = [0, 1, 2]    # <<< seeds used in the paper; [0] trains each model once
if SMOKE_TEST and os.environ.get("DACL_SMOKE_ONE_SEED") != "1":
    E1_SEEDS = cfg.seeds    # the dry run uses two seeds so the ensemble code is tested too
print("Seeds in this round:", E1_SEEDS)

# Seeds in the outer loop: if the session or the compute units run out part-way, every model
# already has the same number of finished seeds, so the comparison stays balanced.
for s in E1_SEEDS:
    for mk in E1_MODELS:
        print(f"=== {mk} | seed {s} ===")
        train_model(mk, s)
unfinished = [(mk, s) for s in E1_SEEDS for mk in E1_MODELS if not run_finished(mk, s)]
print("All training runs finished." if not unfinished else f"Unfinished runs: {unfinished}")

In [ ]:
# CELL 22 · Figure S2: training curves
curve_seed = E1_SEEDS[0]
histories = {}
for mk in E1_MODELS:
    hp = os.path.join(run_dir(mk, curve_seed), "history.csv")
    if os.path.exists(hp):
        histories[mk] = pd.read_csv(hp)

def draw_curves(layout):
    if layout == "horizontal":
        fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.7))
    else:
        fig, axes = plt.subplots(2, 1, figsize=(COL1, 4.9))
    for mk, h in histories.items():
        st = style_of("model", mk)
        axes[0].plot(h.epoch, h.train_loss, color=st["color"], linestyle=st["linestyle"], marker=st["marker"], markevery=max(1, len(h) // 8))
        axes[1].plot(h.epoch, h["val_mIoU@0.5"], color=st["color"], linestyle=st["linestyle"], marker=st["marker"], markevery=max(1, len(h) // 8))
    axes[0].set(title="(a) Training loss", xlabel="Epoch", ylabel="Loss")
    axes[1].set(title="(b) Validation mIoU", xlabel="Epoch", ylabel="mIoU")
    for ax in axes:
        ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    legend_below(fig, [series_handle("model", mk, MODEL_NAMES[mk]) for mk in histories],
                 [MODEL_NAMES[mk] for mk in histories])
    return fig

if histories:
    publish_figure("curves", draw_curves, seed=curve_seed)

### 9.3 Evaluate: tuned thresholds, test and test-clean scores

For each trained run we (1) predict `val` and choose one threshold per class there, (2) apply those thresholds to `test` and `test_clean`. The test set is used only to report, never to choose anything. The **best model for later sections is chosen by validation mIoU**, not by test mIoU, to avoid selection bias.

In [ ]:
# CELL 23 · Evaluate the models and Table 3
E1 = {}   # (model, seed) -> dict with evaluation arrays and thresholds
rows = []
for mk in E1_MODELS:
    for s in E1_SEEDS:
        if not run_finished(mk, s):
            print(f"{mk} seed {s}: training not finished yet, left out of the results"); continue
        model = load_trained(mk, s)
        ev_val = run_eval([model], SPLIT["val"], "single", f"{mk}_s{s}_single_val")
        thr = tune_thresholds(ev_val)
        ev_test = run_eval([model], SPLIT["test"], "single", f"{mk}_s{s}_single_test",
                           conf_thr=np.array(cfg.thr_grid)[thr])
        E1[(mk, s)] = {"val": ev_val, "test": ev_test, "thr": thr}
        r = {"model": mk, "seed": s,
             "val_mIoU": miou(*select(ev_val, thr)),
             "test_mIoU": miou(*select(ev_test, thr)),
             "test_clean_mIoU": miou(*select(ev_test, thr, SPLIT["test_clean"]))}
        r.update({f"IoU_{c}": v for c, v in zip(CLASSES, class_iou(*select(ev_test, thr)))})
        rows.append(r)
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

e1_runs = pd.DataFrame(rows)
e1_runs.to_csv(os.path.join(cfg.project_dir, "tables", "e1_runs.csv"), index=False)

summary = e1_runs.groupby("model")[["val_mIoU", "test_mIoU", "test_clean_mIoU"]].agg(["mean", "std"])
summary.columns = [f"{a} {b}" for a, b in summary.columns]
summary = summary.reindex([m for m in E1_MODELS if m in summary.index])
n_runs = e1_runs.groupby("model").size()
N_SEEDS = int(n_runs.max())

table3 = pd.DataFrame({
    "Model": [MODEL_NAMES[m] for m in summary.index],
    "Seeds": [str(int(n_runs[m])) for m in summary.index],
    "Validation mIoU": [pm(summary.loc[m, "val_mIoU mean"], summary.loc[m, "val_mIoU std"]) for m in summary.index],
    "Test mIoU": [pm(summary.loc[m, "test_mIoU mean"], summary.loc[m, "test_mIoU std"]) for m in summary.index],
    "Leakage-free test mIoU": [pm(summary.loc[m, "test_clean_mIoU mean"], summary.loc[m, "test_clean_mIoU std"])
                               for m in summary.index]})
publish_table("accuracy", table3, raw=summary, **seeds_text(N_SEEDS))

BEST_MODEL = summary["val_mIoU mean"].idxmax()
print("Best model by validation mIoU:", MODEL_NAMES[BEST_MODEL])

In [ ]:
# CELL 24 · Table 4 and Figure 4: per-class IoU
per_class = e1_runs.groupby("model")[[f"IoU_{c}" for c in CLASSES]].mean()
per_class.columns = CLASSES
per_class = per_class.reindex([m for m in E1_MODELS if m in per_class.index])
ref = per_class.loc["fpn_b4"] if "fpn_b4" in per_class.index else per_class.iloc[0]
class_order = list(ref.fillna(-1).sort_values(ascending=False).index)    # best-segmented class first

# ---- Table 4 ----
table4 = pd.DataFrame({"Class": [CLASS_NAMES[c] for c in class_order]})
for m in per_class.index:
    table4[MODEL_NAMES[m]] = [f3(per_class.loc[m, c]) for c in class_order]
mean_row = {"Class": "Mean (mIoU)"}
mean_row.update({MODEL_NAMES[m]: f3(np.nanmean(per_class.loc[m].values)) for m in per_class.index})
table4 = pd.concat([table4, pd.DataFrame([mean_row])], ignore_index=True)
publish_table("per_class", table4, raw=per_class.T, **seeds_text(N_SEEDS))

# ---- Figure 4 ----
def _grouped_barh(ax, classes, models):
    n = len(models); h = 0.8 / n
    ys = np.arange(len(classes))[::-1]           # first class at the top
    for i, m in enumerate(models):
        st = style_of("model", m)
        vals = np.nan_to_num(per_class.loc[m, classes].values.astype(float))
        ax.barh(ys + (n - 1) / 2 * h - i * h, vals, height=h, color=st["color"], hatch=st["hatch"],
                edgecolor="white", linewidth=0.4)
    ax.set_yticks(ys); ax.set_yticklabels([CLASS_NAMES[c] for c in classes])
    ax.set_ylim(-0.6, len(classes) - 0.4)
    ax.set_xlim(0, 1); ax.set_xlabel("Test IoU"); ax.grid(axis="y", visible=False)

def draw_per_class(layout):
    models = list(per_class.index)
    if layout == "horizontal":
        half = math.ceil(len(class_order) / 2)
        fig, axes = plt.subplots(1, 2, figsize=(COL2, 4.6))
        _grouped_barh(axes[0], class_order[:half], models)
        _grouped_barh(axes[1], class_order[half:], models)
        axes[1].set_ylim(-0.6 - (half - len(class_order[half:])), len(class_order[half:]) - 0.4)
    else:
        fig, ax = plt.subplots(1, 1, figsize=(COL1, 8.0))
        _grouped_barh(ax, class_order, models)
    legend_below(fig, [series_handle("model", m, MODEL_NAMES[m], bar=True) for m in models],
                 [MODEL_NAMES[m] for m in models], ncol=2 if layout == "vertical" else 4)
    return fig

publish_figure("per_class", draw_per_class, **seeds_text(N_SEEDS))

### 9.4 Is the difference real? Paired bootstrap over test images

Seed-to-seed variation and the particular test images both add noise. For each model we compare against the replicated baseline `fpn_b4` with a **paired bootstrap**: resample test images with replacement (the same images for both models), recompute each model's mIoU (averaged over its seeds) and record the difference. The 2.5th and 97.5th percentiles give a 95% confidence interval.

Because three comparisons are made, p-values are adjusted with **Holm's method**. Claim an improvement only when the adjusted p-value is below 0.05 *and* the interval excludes zero.

In [ ]:
# CELL 25 · Table 5: paired bootstrap tests
def stacked_IU(mk, names=None):
    Is, Us = [], []
    for s in E1_SEEDS:
        if (mk, s) not in E1: continue
        I, U = select(E1[(mk, s)]["test"], E1[(mk, s)]["thr"], names)
        Is.append(I); Us.append(U)
    return np.stack(Is), np.stack(Us)          # (S, N, C)

def boot_miou(I, U, idx):
    """I, U: (S, N, C); idx: (B, N) resampled image indices -> (B,) seed-mean mIoU."""
    Ib, Ub = I[:, idx].sum(2), U[:, idx].sum(2)                          # (S, B, C)
    iou = np.where(Ub > 0, Ib / np.maximum(Ub, 1), np.nan)              # a class absent from a resample is skipped, not scored 0
    return np.nanmean(np.nanmean(iou, axis=2), axis=0)

N_BOOT = 5000 if not SMOKE_TEST else 200

def paired_bootstrap(mk_a, mk_b, names=None, n_boot=N_BOOT, chunk=250, seed=0):
    Ia, Ua = stacked_IU(mk_a, names); Ib, Ub = stacked_IU(mk_b, names)
    N = Ia.shape[1]; rng = np.random.default_rng(seed); diffs = []
    for k in range(0, n_boot, chunk):
        idx = rng.integers(0, N, size=(min(chunk, n_boot - k), N))
        diffs.append(boot_miou(Ia, Ua, idx) - boot_miou(Ib, Ub, idx))
    d = np.concatenate(diffs)
    obs = boot_miou(Ia, Ua, np.arange(N)[None])[0] - boot_miou(Ib, Ub, np.arange(N)[None])[0]
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return obs, np.percentile(d, 2.5), np.percentile(d, 97.5), p

def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); running = 0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i]); adj[i] = min(1.0, running)
    return adj

baseline = "fpn_b4"
# (1) every model against the replicated baseline; (2) the best model (chosen on validation) against each other model
pairs = [(mk, baseline) for mk in E1_MODELS if mk != baseline and mk in summary.index]
pairs += [(BEST_MODEL, mk) for mk in E1_MODELS if mk not in (BEST_MODEL, baseline) and mk in summary.index]
comp = []
for mk, ref in pairs:
    for subset_name, names in [("test", None), ("test_clean", SPLIT["test_clean"])]:
        obs, lo, hi, p = paired_bootstrap(mk, ref, names)
        comp.append({"model": mk, "vs": ref, "subset": subset_name, "delta_mIoU": obs,
                     "CI95_low": lo, "CI95_high": hi, "p": p})
comp = pd.DataFrame(comp)
if len(comp):
    comp["p_holm"] = np.nan
    for sub in comp.subset.unique():
        mask = comp.subset == sub
        comp.loc[mask, "p_holm"] = holm(comp.loc[mask, "p"].values)
    comp["significant"] = (comp.p_holm < 0.05) & ((comp.CI95_low > 0) | (comp.CI95_high < 0))
if len(comp):
    table5 = pd.DataFrame({
        "Comparison": [f"{MODEL_NAMES[m]} vs {MODEL_NAMES[r]}" for m, r in zip(comp.model, comp.vs)],
        "Test set": ["Test" if sname == "test" else "Leakage-free test" for sname in comp.subset],
        "ΔmIoU": [f"{v:+.3f}" for v in comp.delta_mIoU],
        "95% CI": [f"[{lo:+.3f}, {hi:+.3f}]" for lo, hi in zip(comp.CI95_low, comp.CI95_high)],
        "p (Holm)": [pfmt(p) for p in comp.p_holm],
        "Significant": ["Yes" if v else "No" for v in comp.significant]})
    publish_table("bootstrap", table5, raw=comp, baseline=MODEL_NAMES[baseline], best=MODEL_NAMES[BEST_MODEL],
                  n_boot=f"{N_BOOT:,}", **seeds_text(N_SEEDS))

### 9.5 Optional E2: does input resolution limit thin defects?

Cracks, exposed rebars and cavities are a few pixels wide even at 512 px, and DINOv2 sees the image through 14 px patches. This experiment trains the best model at **1024 px** (seed 0) and compares it with the 512 px model on the *same* 1024 px ground truth: the 512 px model's predictions are upsampled, so both are judged against the finer masks.

It roughly quadruples memory and time per image (batch size is divided by 4 and gradients are accumulated to keep the effective batch size), so it is **off by default**. Set `RUN_E2 = True` once E1 is complete.

In [ ]:
# CELL 26 · Optional E2: resolution study and Table S3
RUN_E2 = SMOKE_TEST          # set True for the real run once E1 is done
E2_SIZE = 2 * cfg.img_size
THIN = ["Crack", "ExposedRebars", "Cavity", "JTape", "Restformwork"]

if RUN_E2:
    build_cache(E2_SIZE, splits=("train", "validation"))
    train_model(BEST_MODEL, E1_SEEDS[0], size=E2_SIZE)
    m_hi = load_trained(BEST_MODEL, E1_SEEDS[0], size=E2_SIZE)
    m_lo = load_trained(BEST_MODEL, E1_SEEDS[0])
    res = {}
    for label, m, inp in [(str(cfg.img_size), m_lo, cfg.img_size), (str(E2_SIZE), m_hi, E2_SIZE)]:
        ev_v = run_eval([m], SPLIT["val"], "single", f"e2_{BEST_MODEL}_{label}_val", size=E2_SIZE, input_size=inp)
        ev_t = run_eval([m], SPLIT["test"], "single", f"e2_{BEST_MODEL}_{label}_test", size=E2_SIZE, input_size=inp)
        th = tune_thresholds(ev_v)
        res[label] = class_iou(*select(ev_t, th))
    e2 = pd.DataFrame(res, index=CLASSES)
    e2["gain"] = e2.iloc[:, 1] - e2.iloc[:, 0]
    lo_col, hi_col = f"{cfg.img_size} px input", f"{E2_SIZE} px input"
    e2_order = list(e2["gain"].fillna(-9).sort_values(ascending=False).index)
    table_s3 = pd.DataFrame({"Class": [CLASS_NAMES[c] for c in e2_order],
                             lo_col: [f3(e2.loc[c].iloc[0]) for c in e2_order],
                             hi_col: [f3(e2.loc[c].iloc[1]) for c in e2_order],
                             "Gain": [("–" if not np.isfinite(e2.loc[c, "gain"]) else f"{e2.loc[c, 'gain']:+.3f}") for c in e2_order]})
    table_s3 = pd.concat([table_s3, pd.DataFrame([{"Class": "Mean (mIoU)", lo_col: f3(np.nanmean(e2.iloc[:, 0])),
                                                   hi_col: f3(np.nanmean(e2.iloc[:, 1])),
                                                   "Gain": f"{np.nanmean(e2.iloc[:, 1]) - np.nanmean(e2.iloc[:, 0]):+.3f}"}])],
                         ignore_index=True)
    publish_table("resolution", table_s3, raw=e2, model=MODEL_NAMES[BEST_MODEL], size=E2_SIZE)
    del m_hi, m_lo
else:
    print("E2 skipped (RUN_E2 = False).")

### 9.6 Optional: is 30 epochs enough? (training-length check)

Reviewers often ask whether the models were simply trained too briefly. This cell answers that directly: it retrains **one model, one seed** for **twice as many epochs (60) without early stopping**, giving the longer schedule every chance, and compares it with the 30-epoch protocol on the same test images. By default it retrains **full fine-tuning** (its best epochs came latest) with its **weakest seed**, because that is where a longer schedule is most likely to help.

- The comparison is **paired** (same test images, same per-class threshold tuning on validation) and uses the same **bootstrap** as Table 5, so the difference comes with a 95% confidence interval and a p-value.
- When round 2 (three seeds) is done, the cell also reports the **seed-to-seed standard deviation**: a difference smaller than that is practically irrelevant.
- The longer run is stored in its own folder (`runs/<model>_ep60_s0_512px`), so the 30-epoch results are never overwritten. Like CELL 21, it resumes after a disconnect.

It costs one extra training run (about 2 h for DINOv2 on an A100 with 8 image loaders), so it is off in the earlier notebooks; **in this final notebook it is already switched on**.

In [ ]:
# CELL 26b · Optional: training-length check, Table S5 and Figure S3
RUN_EPOCH_CHECK = True           # <<< switched on for the final run (about 2 h)
EPOCH_CHECK_MODEL = "dino_full"  # full fine-tuning: its best epochs came latest, so it is the model a reviewer would question
# the seed with the lowest validation mIoU, i.e. the run that most plausibly needed more epochs
_runs = e1_runs[e1_runs.model == EPOCH_CHECK_MODEL]
EPOCH_CHECK_SEED = int(_runs.sort_values("val_mIoU").seed.iloc[0]) if len(_runs) else E1_SEEDS[0]
LONG_EPOCHS = 2 * cfg.epochs     # 60 in the real run

# Register the table and figure numbers (also works if CELL 05 was run before this cell existed)
TABLES.setdefault("epochs", ("Table S5", "Effect of training length, {model} (seed {seed}): the {short}-epoch protocol used for all models against a {long}-epoch schedule without early stopping. The last row gives the paired bootstrap difference ({n_boot} resamples of test images) with its 95% confidence interval.{sd_note}"))
FIGURES.setdefault("epochs", ("Figure S3", "Training-length check, {model} (seed {seed}): (a) training loss and (b) validation mIoU at a threshold of 0.5 for the {short}-epoch and {long}-epoch schedules. The dotted line marks the end of the {short}-epoch schedule."))

if not RUN_EPOCH_CHECK:
    print("Training-length check skipped (RUN_EPOCH_CHECK = False).")
elif (EPOCH_CHECK_MODEL, EPOCH_CHECK_SEED) not in E1:
    raise RuntimeError(f"Run CELLS 21-25 first: the {cfg.epochs}-epoch run of {EPOCH_CHECK_MODEL} seed {EPOCH_CHECK_SEED} is not evaluated yet.")
else:
    long_key = f"{EPOCH_CHECK_MODEL}_ep{LONG_EPOCHS}"
    MODEL_BUILDERS[long_key] = MODEL_BUILDERS[EPOCH_CHECK_MODEL]   # same architecture, separate run folder
    h_short = pd.read_csv(os.path.join(run_dir(EPOCH_CHECK_MODEL, EPOCH_CHECK_SEED), "history.csv"))
    print(f"Retraining {MODEL_NAMES[EPOCH_CHECK_MODEL]} (seed {EPOCH_CHECK_SEED}) for {LONG_EPOCHS} epochs; "
          f"estimated time about {h_short.minutes.mean() * LONG_EPOCHS / 60:.1f} h.")

    # ---- train: no early stopping, so the longer schedule gets its full chance ----
    old_patience = cfg.patience
    cfg.patience = LONG_EPOCHS + 1
    try:
        train_model(long_key, EPOCH_CHECK_SEED, epochs=LONG_EPOCHS)
    finally:
        cfg.patience = old_patience
    h_long = pd.read_csv(os.path.join(run_dir(long_key, EPOCH_CHECK_SEED), "history.csv"))

    # ---- evaluate exactly as in CELL 23: thresholds tuned on validation, applied to test ----
    m_long = load_trained(long_key, EPOCH_CHECK_SEED)
    ev_v = run_eval([m_long], SPLIT["val"], "single", f"{long_key}_s{EPOCH_CHECK_SEED}_single_val")
    thr_l = tune_thresholds(ev_v)
    ev_t = run_eval([m_long], SPLIT["test"], "single", f"{long_key}_s{EPOCH_CHECK_SEED}_single_test",
                    conf_thr=np.array(cfg.thr_grid)[thr_l])
    del m_long
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    MODEL_BUILDERS.pop(long_key, None)

    short = E1[(EPOCH_CHECK_MODEL, EPOCH_CHECK_SEED)]
    def scores(ev_val, ev_test, thr):
        return {"val": miou(*select(ev_val, thr)), "test": miou(*select(ev_test, thr)),
                "clean": miou(*select(ev_test, thr, SPLIT["test_clean"]))}
    s_short, s_long = scores(short["val"], short["test"], short["thr"]), scores(ev_v, ev_t, thr_l)

    # ---- paired bootstrap of the test mIoU difference (same resampled images for both runs) ----
    assert (ev_t["names"] == short["test"]["names"]).all(), "test images are not in the same order"
    Il, Ul = select(ev_t, thr_l)
    Is_, Us_ = select(short["test"], short["thr"])
    rng = np.random.default_rng(0); N = Il.shape[0]; diffs = []
    for k in range(0, N_BOOT, 250):
        idx = rng.integers(0, N, size=(min(250, N_BOOT - k), N))
        diffs.append(boot_miou(Il[None], Ul[None], idx) - boot_miou(Is_[None], Us_[None], idx))
    d = np.concatenate(diffs)
    obs = s_long["test"] - s_short["test"]
    lo, hi = np.percentile(d, 2.5), np.percentile(d, 97.5)
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    p_txt = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

    runs_m = e1_runs[e1_runs.model == EPOCH_CHECK_MODEL]
    seed_sd = float(runs_m.test_mIoU.std()) if len(runs_m) >= 2 else float("nan")
    sd_note = (f" For comparison, the seed-to-seed standard deviation of test mIoU is {seed_sd:.3f} ({len(runs_m)} seeds)."
               if np.isfinite(seed_sd) else "")

    best_epoch = lambda h: int(h.loc[h["val_mIoU@0.5"].idxmax(), "epoch"])
    table_s5 = pd.DataFrame([
        [f"{cfg.epochs} epochs, early stopping (protocol)", str(len(h_short)), str(best_epoch(h_short)),
         f3(s_short["val"]), f3(s_short["test"]), f3(s_short["clean"])],
        [f"{LONG_EPOCHS} epochs, no early stopping", str(len(h_long)), str(best_epoch(h_long)),
         f3(s_long["val"]), f3(s_long["test"]), f3(s_long["clean"])],
        [f"Difference ({LONG_EPOCHS} vs {cfg.epochs} epochs)", "–", "–",
         f"{s_long['val'] - s_short['val']:+.3f}", f"{obs:+.3f} [{lo:+.3f}, {hi:+.3f}], {p_txt}",
         f"{s_long['clean'] - s_short['clean']:+.3f}"],
    ], columns=["Schedule", "Epochs trained", "Best epoch", "Validation mIoU", "Test mIoU", "Leakage-free test mIoU"])
    publish_table("epochs", table_s5, raw=pd.DataFrame({"short": s_short, "long": s_long}),
                  model=MODEL_NAMES[EPOCH_CHECK_MODEL], seed=EPOCH_CHECK_SEED, short=cfg.epochs, long=LONG_EPOCHS,
                  n_boot=f"{N_BOOT:,}", sd_note=sd_note)

    # ---- Figure S3: both learning curves on one epoch axis ----
    series = [(h_short, f"{cfg.epochs} epochs (protocol)", PALETTE[0], "-", "o"),
              (h_long, f"{LONG_EPOCHS} epochs", PALETTE[1], "--", "s")]
    def draw_epochs(layout):
        if layout == "horizontal":
            fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.7))
        else:
            fig, axes = plt.subplots(2, 1, figsize=(COL1, 4.9))
        for h, _, col, ls, mk in series:
            me = max(1, len(h) // 8)
            axes[0].plot(h.epoch, h.train_loss, color=col, linestyle=ls, marker=mk, markevery=me)
            axes[1].plot(h.epoch, h["val_mIoU@0.5"], color=col, linestyle=ls, marker=mk, markevery=me)
        for ax in axes:
            ax.axvline(cfg.epochs, color=INK2, linestyle=":", linewidth=0.8)
            ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
        axes[0].set(title="(a) Training loss", xlabel="Epoch", ylabel="Loss")
        axes[1].set(title="(b) Validation mIoU", xlabel="Epoch", ylabel="mIoU")
        legend_below(fig, [Line2D([], [], color=c, linestyle=ls, marker=m) for _, _, c, ls, m in series],
                     [s[1] for s in series])
        return fig
    publish_figure("epochs", draw_epochs, model=MODEL_NAMES[EPOCH_CHECK_MODEL], seed=EPOCH_CHECK_SEED,
                   short=cfg.epochs, long=LONG_EPOCHS);

    # ---- plain-language verdict and a sentence for the paper ----
    significant = not (lo <= 0 <= hi)
    print(f"\nTest mIoU: {s_short['test']:.3f} ({cfg.epochs} epochs) vs {s_long['test']:.3f} ({LONG_EPOCHS} epochs); "
          f"difference {obs:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}], {p_txt}")
    if not significant:
        extra = (f" and smaller than the seed-to-seed standard deviation ({seed_sd:.3f})"
                 if np.isfinite(seed_sd) and abs(obs) < seed_sd else "")
        print(f"Verdict: longer training gives no significant gain; the {cfg.epochs}-epoch protocol is sufficient.")
        print(f'Suggested sentence: "Extending training of {MODEL_NAMES[EPOCH_CHECK_MODEL]} from {cfg.epochs} to '
              f'{LONG_EPOCHS} epochs without early stopping changed test mIoU by {obs:+.3f} '
              f'(95% CI {lo:+.3f} to {hi:+.3f}), which is not significant{extra}; '
              f'all models were therefore trained for {cfg.epochs} epochs (Table S5, Figure S3)."')
    elif obs > 0:
        print("Verdict: longer training helps significantly. Either train all four models for the longer schedule "
              "(so the comparison stays fair) or state this as a limitation in the paper.")
    else:
        print(f"Verdict: longer training is significantly worse (overfitting); the {cfg.epochs}-epoch protocol with early "
              "stopping is the better choice. Report this in the paper as support for the protocol.")

---
## 10. Experiment E3: which uncertainty estimate is best calibrated? (RQ3)

A model is **calibrated** if, among all pixels where it says "70% Spalling", about 70% really are Spalling. Inspectors can only use probabilities as probabilities when this holds.

Four estimators are compared on the best model from E1:

| Method | How it works | Cost per image |
|---|---|---|
| `single` | One deterministic pass | 1 pass |
| `mc` | **MC dropout**: keep dropout active at test time, average `mc_samples` passes | 20 passes |
| `tta` | **Test-time augmentation**: average over scales × horizontal flip | 6 passes |
| `ensemble` | **Deep ensemble**: average the models trained with different seeds | 1 pass × seeds |

Each is then **temperature-scaled per class**: a single scalar $T_c$ is fitted on the `val` pixels to minimise cross-entropy, and test probabilities become $\sigma(\text{logit}(\bar p)/T_c)$. $T_c > 1$ means the model was over-confident. Temperature scaling is monotonic, so it never changes the predicted masks or the IoU.

**Metrics** (per class, then averaged): expected calibration error (**ECE**, 15 equal-width bins), **Brier score** and **negative log-likelihood (NLL)**. Because most sampled pixels are confident background, plain ECE is dominated by easy negatives; we therefore also report **ECE on active pixels**, restricted to pixels that are positive or predicted with p ≥ 0.05.

If MC dropout clearly lowers mIoU compared with `single`, the dropout layers are too strong for test-time sampling; report this as a finding rather than tuning the rate on the test set.

A caveat for the paper: MC dropout's strength depends on where dropout sits in each architecture (LoRA dropout and a decoder dropout for `dino_lora`; decoder dropout only for `fpn_b4`), so it is not an identical treatment across models.

In [ ]:
# CELL 27 · Run the four uncertainty methods (E3)
UNC_MODEL = BEST_MODEL
unc_seeds = [s for s in E1_SEEDS if run_finished(UNC_MODEL, s)]
seed_models = [load_trained(UNC_MODEL, s) for s in unc_seeds]
METHODS = {"single": seed_models[:1], "mc": seed_models[:1], "tta": seed_models[:1], "ensemble": seed_models}
if len(seed_models) < 2:
    METHODS.pop("ensemble")
    print("Only one trained seed of", MODEL_NAMES[UNC_MODEL], "-> the deep ensemble is skipped. "
          "Train more seeds (round 2) to include it.")

UNC = {}
for meth, ms in METHODS.items():
    tag = f"e3_{UNC_MODEL}_{meth}" + (f"{len(ms)}" if meth == "ensemble" else "")   # ensemble cache depends on the number of seeds
    ev_v = run_eval(ms, SPLIT["val"], meth, f"{tag}_val", samples=True)
    thr = tune_thresholds(ev_v)
    ev_t = run_eval(ms, SPLIT["test"], meth, f"{tag}_test", conf_thr=np.array(cfg.thr_grid)[thr],
                    samples=True)
    UNC[meth] = {"val": ev_v, "test": ev_t, "thr": thr}
    print(f"{METHOD_NAMES[meth]:24s} test mIoU = {miou(*select(ev_t, thr)):.4f}")

In [ ]:
# CELL 28 · Table 6 and Table S2: calibration
def fit_temperature(p, y, min_pos=20):
    """Per-class temperature by LBFGS on BCE; falls back to T=1 when a class has too few positives."""
    p = torch.as_tensor(p, dtype=torch.float64).clamp(EPS, 1 - EPS)
    y = torch.as_tensor(y, dtype=torch.float64)
    if y.sum() < min_pos:
        return 1.0
    logit = torch.log(p) - torch.log1p(-p)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.5, max_iter=100, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(logit / log_t.exp(), y)
        loss.backward(); return loss
    opt.step(closure)
    return float(log_t.exp().clamp(0.05, 20.0))

def apply_temperature(p, T):
    p = np.clip(p.astype(np.float64), EPS, 1 - EPS)
    return 1 / (1 + np.exp(-(np.log(p) - np.log1p(-p)) / T))

def ece(p, y, bins=15):
    if len(p) == 0: return np.nan
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    total = 0.0
    for b in range(bins):
        sel = idx == b
        if sel.any(): total += sel.mean() * abs(y[sel].mean() - p[sel].mean())
    return total

def calib_metrics(p, y):
    p = np.clip(p.astype(np.float64), EPS, 1 - EPS); y = y.astype(np.float64)
    active = (y > 0) | (p >= 0.05)
    return {"ECE": ece(p, y), "ECE_active": ece(p[active], y[active]), "Brier": np.mean((p - y) ** 2),
            "NLL": -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))}

calib_rows, TEMPS = [], {}
for meth, d in UNC.items():
    pv = d["val"]["p"].astype(np.float32).transpose(1, 0, 2).reshape(C, -1)
    yv = d["val"]["y"].transpose(1, 0, 2).reshape(C, -1)
    pt = d["test"]["p"].astype(np.float32).transpose(1, 0, 2).reshape(C, -1)
    yt = d["test"]["y"].transpose(1, 0, 2).reshape(C, -1)
    TEMPS[meth] = np.array([fit_temperature(pv[c], yv[c]) for c in range(C)])
    d["p_test_cal"] = np.stack([apply_temperature(pt[c], TEMPS[meth][c]) for c in range(C)])
    for stage, probs in [("raw", pt), ("temp-scaled", d["p_test_cal"])]:
        per = pd.DataFrame([calib_metrics(probs[c], yt[c]) for c in range(C) if yt[c].sum() > 0])
        calib_rows.append({"method": meth, "stage": stage, "test_mIoU": miou(*select(d["test"], d["thr"])),
                           **per.mean().to_dict()})

calib = pd.DataFrame(calib_rows)
table6 = pd.DataFrame({
    "Method": [METHOD_NAMES[m] for m in calib.method],
    "Calibration": ["None" if st == "raw" else "Temperature scaling" for st in calib.stage],
    "mIoU": [f3(v) for v in calib.test_mIoU], "ECE": [f3(v, 4) for v in calib.ECE],
    "ECE (active)": [f3(v, 4) for v in calib.ECE_active], "Brier": [f3(v, 4) for v in calib.Brier],
    "NLL": [f3(v, 4) for v in calib.NLL]})
# caption label: the single model, MC dropout and TTA use the first seed; the ensemble combines all seeds
UNC_LABEL = (f"{MODEL_NAMES[UNC_MODEL]} (seed {unc_seeds[0]}; the deep ensemble combines seeds {join_and(unc_seeds)})"
             if "ensemble" in METHODS else f"{MODEL_NAMES[UNC_MODEL]} (seed {unc_seeds[0]})")
publish_table("calibration", table6, raw=calib, model=UNC_LABEL)

temps = pd.DataFrame(TEMPS, index=CLASSES)
table_s2 = pd.DataFrame({"Class": [CLASS_NAMES[c] for c in CLASSES]})
for m in temps.columns:
    table_s2[METHOD_NAMES[m]] = [f"{v:.2f}" for v in temps[m]]
publish_table("temperatures", table_s2, raw=temps, model=UNC_LABEL)

In [ ]:
# CELL 29 · Figure 5: reliability diagrams
def reliability_curve(p, y, bins=10, min_count=20):
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    conf, acc = [], []
    for b in range(bins):
        sel = idx == b
        if sel.sum() >= min_count:
            conf.append(p[sel].mean()); acc.append(y[sel].mean())
    return np.array(conf), np.array(acc)

rel_classes = ["Spalling", "Crack", "Rust"]

def draw_reliability(layout):
    if layout == "horizontal":
        fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.9))
    else:
        fig, axes = plt.subplots(3, 1, figsize=(COL1, 8.2))
    for k, (ax, cname) in enumerate(zip(axes, rel_classes)):
        c = CLASS_INDEX[cname]
        ax.plot([0, 1], [0, 1], linestyle=(0, (4, 3)), linewidth=0.8, color=INK2)
        for meth, d in UNC.items():
            st = style_of("method", meth)
            yt = d["test"]["y"][:, c].reshape(-1)
            cf, ac = reliability_curve(d["p_test_cal"][c], yt)
            ax.plot(cf, ac, color=st["color"], marker=st["marker"], linestyle=st["linestyle"])
        ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Predicted probability", ylabel="Observed frequency")
        ax.set_title(f"({'abc'[k]}) {CLASS_NAMES[cname]}")
        ax.set_box_aspect(1)
    handles = [series_handle("method", m, METHOD_NAMES[m]) for m in UNC] + \
              [Line2D([], [], color=INK2, linestyle=(0, (4, 3)), linewidth=0.8, label="Perfect calibration")]
    legend_below(fig, handles, [h.get_label() for h in handles], ncol=5 if layout == "horizontal" else 2)
    return fig

publish_figure("reliability", draw_reliability, model=UNC_LABEL,
               c1=CLASS_NAMES[rel_classes[0]].lower(), c2=CLASS_NAMES[rel_classes[1]].lower(),
               c3=CLASS_NAMES[rel_classes[2]].lower())

---
## 11. Experiment E4: does uncertainty point the inspector to the errors? (RQ4)

Calibration says whether probabilities are honest *on average*. An inspector needs something sharper: **if I only have time to check 10% of the image, does the uncertainty map send me to the mistakes?**

For every sampled test pixel we mark an **error** when the thresholded prediction disagrees with the label, then rank pixels from most to least uncertain (binary entropy of the calibrated probability):

- **Error recall at budget *b*:** the share of all errors found when reviewing the top *b* fraction of pixels. Random review finds a share of exactly *b*, so anything above the diagonal is useful.
- **AURC** (area under the risk–coverage curve): keep the most *confident* pixels first and track the error rate of what is kept. Lower is better.

Two views are reported. The **per-class view** treats each class channel separately. The **inspector view** treats a pixel as wrong if *any* class is wrong and scores it by its *highest* class uncertainty, which matches how a person would read a single combined uncertainty map.

In [ ]:
# CELL 30 · Table 7 and Figure 6: uncertainty-guided error review
BUDGETS = [0.05, 0.10, 0.20]

def error_recall_curve(score, err, n_pts=101):
    order = np.argsort(-score, kind="stable")
    cum = np.cumsum(err[order]) / max(err.sum(), 1)
    xs = np.linspace(0, 1, n_pts)
    return xs, np.interp(xs * len(err), np.arange(1, len(err) + 1), cum, left=0)

def aurc(score, err):
    order = np.argsort(score, kind="stable")            # most confident first
    risk = np.cumsum(err[order]) / np.arange(1, len(err) + 1)
    return float(risk.mean())

def recall_at(score, err, b):
    k = max(1, int(round(b * len(err))))
    top = np.argsort(-score, kind="stable")[:k]
    return float(err[top].sum() / max(err.sum(), 1))

e4_rows, curves = [], {}
for meth, d in UNC.items():
    thr = np.array(cfg.thr_grid)[d["thr"]]
    p_raw = d["test"]["p"].astype(np.float32)                         # (N, C, K)
    y = d["test"]["y"]
    p_cal = d["p_test_cal"].reshape(C, p_raw.shape[0], -1).transpose(1, 0, 2)
    err = (p_raw > thr[None, :, None]) != y                          # (N, C, K)
    unc = bin_entropy(torch.from_numpy(p_cal)).numpy()
    # per-class view
    per = []
    for c in range(C):
        e_c, u_c = err[:, c].reshape(-1), unc[:, c].reshape(-1)
        if e_c.sum() < 10: continue
        per.append({"AURC": aurc(u_c, e_c), **{f"recall@{int(b*100)}%": recall_at(u_c, e_c, b) for b in BUDGETS}})
    per = pd.DataFrame(per).mean()
    # inspector view
    e_any, u_max = err.any(1).reshape(-1), unc.max(1).reshape(-1)
    insp = {"AURC": aurc(u_max, e_any), **{f"recall@{int(b*100)}%": recall_at(u_max, e_any, b) for b in BUDGETS}}
    curves[meth] = error_recall_curve(u_max, e_any)
    e4_rows.append({"method": meth, "view": "per-class (mean)", **per.to_dict()})
    e4_rows.append({"method": meth, "view": "inspector (any class)", "pixel_error_rate": float(e_any.mean()), **insp})

e4 = pd.DataFrame(e4_rows)
rows7 = []
for view in ["inspector (any class)", "per-class (mean)"]:
    for _, r in e4[e4.view == view].iterrows():
        rows7.append({"Method": METHOD_NAMES[r["method"]],
                      "View": "Inspector (any class)" if view.startswith("inspector") else "Per class (mean)",
                      "AURC": f3(r["AURC"], 4), "Errors found, 5%": f3(r["recall@5%"]),
                      "Errors found, 10%": f3(r["recall@10%"]), "Errors found, 20%": f3(r["recall@20%"])})
    if view.startswith("inspector"):
        rows7.append({"Method": "Random review", "View": "Inspector (any class)", "AURC": "–",
                      "Errors found, 5%": "0.050", "Errors found, 10%": "0.100", "Errors found, 20%": "0.200"})
publish_table("error_review", pd.DataFrame(rows7), raw=e4, model=UNC_LABEL)

def draw_error_recall(layout):
    fig, ax = plt.subplots(1, 1, figsize=(COL15, 2.9) if layout == "horizontal" else (COL1, 3.9))
    for b in BUDGETS:
        ax.axvline(b, color=GRID, linewidth=0.8, zorder=0)
    ax.plot([0, 1], [0, 1], linestyle=(0, (4, 3)), linewidth=0.8, color=INK2)
    for meth, (xs, rec) in curves.items():
        st = style_of("method", meth)
        ax.plot(xs, rec, color=st["color"], linestyle=st["linestyle"], marker=st["marker"], markevery=10)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Share of pixels reviewed (most uncertain first)",
           ylabel="Share of errors found")
    if layout == "vertical":
        ax.set_box_aspect(1)
    handles = [series_handle("method", m, METHOD_NAMES[m]) for m in curves] + \
              [Line2D([], [], color=INK2, linestyle=(0, (4, 3)), linewidth=0.8, label="Random review")]
    if layout == "horizontal":
        fig.legend(handles, [h.get_label() for h in handles], loc="outside right center", ncol=1)
    else:
        legend_below(fig, handles, [h.get_label() for h in handles], ncol=2)
    return fig

publish_figure("error_recall", draw_error_recall, model=UNC_LABEL)

---
## 12. Experiment E5: are the explanations faithful? (RQ2)

### 12.1 What is explained

For segmentation there is no single output to explain. We explain a **region score**: the mean predicted probability of class *c* inside the predicted region *M* (falling back to the ground-truth region when the prediction is nearly empty). The question becomes: *which pixels make the model believe this region is Spalling?*

Images are drawn from the test set, targeting the classes the dacl10k authors found hardest to separate (Spalling, Rockpocket, Washouts/Concrete corrosion, Weathering, Wetspot, Crack, Alligator Crack).

### 12.2 Two explanation methods

- **Seg-Grad-CAM** (gradient-based): weights the feature maps of a late decoder layer by the gradient of the region score. Cheap (one backward pass) but known to be sometimes unfaithful.
- **Occlusion** (perturbation-based): replaces one square patch at a time with a blurred version and records how much the region score drops. Slow (one pass per patch) but directly causal.

### 12.3 Three tests every map must pass before it is interpreted

| Test | Procedure | A faithful map shows |
|---|---|---|
| **Deletion** | Replace pixels with blur in order of attribution, highest first; record the region score | A fast drop: **low** area under the curve |
| **Insertion** | Start from the blurred image and restore pixels in order of attribution | A fast rise: **high** area under the curve |
| **Model randomisation** (Adebayo et al., 2018) | Recompute the map on a copy of the model with randomised weights | **Low** rank correlation with the trained model's map |

Both curve tests are compared with a **random pixel order**; a method that does not beat random ordering is not explaining the model. We also report **off-target leakage**: the share of positive attribution lying outside the region *M*. High leakage means the decision relies on context, which is sometimes legitimate (wet spots next to drainage) and sometimes a shortcut.

Occlusion is the expensive step: each image needs (512 / 32)² = 256 forward passes, done twice (trained and randomised model). Time the first few images and reduce `n_xai_images` if needed.

In [ ]:
# CELL 31 · Explanation functions
from torchvision.transforms.functional import gaussian_blur
trapz = getattr(np, "trapezoid", None) or np.trapz

xai_model = load_trained(BEST_MODEL, E1_SEEDS[0])
xai_thr = np.array(cfg.thr_grid)[E1[(BEST_MODEL, E1_SEEDS[0])]["thr"]]
TARGET_CLASSES = sorted({c for pair in CONFUSABLE_PAIRS for c in pair}, key=CLASS_INDEX.get)

def blurred(x):
    k = int(2 * round(0.1 * x.shape[-1]) + 1)
    return gaussian_blur(x, kernel_size=[k, k], sigma=[k / 3.0, k / 3.0])

@torch.no_grad()
def region_score(model, xb, c, M, bs=16):
    out = []
    for k in range(0, len(xb), bs):
        p = torch.sigmoid(model(xb[k:k + bs])["logits"][:, c].float())
        out.append((p * M).sum(dim=(1, 2)) / M.sum())
    return torch.cat(out)

def seg_gradcam(model, x, c, M):
    acts = {}
    h = model.cam_layer.register_forward_hook(lambda mod, inp, out: acts.__setitem__("a", out))
    with torch.enable_grad():
        logits = model(x)["logits"].float()
        score = (logits[0, c] * M).sum() / M.sum()
        grad = torch.autograd.grad(score, acts["a"])[0]
    h.remove()
    a = acts["a"].detach()
    cam = F.relu((grad.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return cam / (cam.max() + 1e-8)

@torch.no_grad()
def occlusion_map(model, x, c, M, base, patch):
    H, W = x.shape[-2:]
    s0 = region_score(model, x, c, M)[0]
    coords = [(i, j) for i in range(0, H, patch) for j in range(0, W, patch)]
    attr = torch.zeros(H, W, device=x.device)
    for k in range(0, len(coords), 16):
        chunk = coords[k:k + 16]
        xb = x.repeat(len(chunk), 1, 1, 1)
        for b, (i, j) in enumerate(chunk):
            xb[b, :, i:i + patch, j:j + patch] = base[0, :, i:i + patch, j:j + patch]
        s = region_score(model, xb, c, M)
        for b, (i, j) in enumerate(chunk):
            attr[i:i + patch, j:j + patch] = s0 - s[b]
    return attr

@torch.no_grad()
def deletion_insertion(model, x, base, attr, c, M, steps=20):
    H, W = x.shape[-2:]
    order = torch.argsort(attr.flatten(), descending=True)
    fracs = np.linspace(0, 1, steps + 1)
    dels, ins = [], []
    for f in fracs:
        m = torch.zeros(H * W, device=x.device); m[order[: int(round(f * H * W))]] = 1
        m = m.view(1, 1, H, W)
        dels.append(x * (1 - m) + base * m); ins.append(base * (1 - m) + x * m)
    sd = region_score(model, torch.cat(dels), c, M).cpu().numpy()
    si = region_score(model, torch.cat(ins), c, M).cpu().numpy()
    return float(trapz(sd, fracs)), float(trapz(si, fracs)), sd, si

def randomized_copy(model, seed=123):
    g = torch.Generator().manual_seed(seed)
    rm = copy.deepcopy(model)
    with torch.no_grad():
        for p in rm.parameters():
            if p.numel() > 1:
                p.copy_((torch.randn(p.shape, generator=g) * (p.float().std().item() + 1e-3)).to(p.dtype))
    return rm.eval()

def rank_corr(a, b, size=64):
    a = F.adaptive_avg_pool2d(a[None, None].float(), size).flatten().cpu().numpy()
    b = F.adaptive_avg_pool2d(b[None, None].float(), size).flatten().cpu().numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # a constant map has no defined rank correlation
        r = spearmanr(a, b).correlation
    return float(r) if np.isfinite(r) else 0.0

In [ ]:
# CELL 32 · Run the explanations (E5)
# Pick test images whose ground truth contains a confusable class, STRATIFIED by class:
# candidates are collected per target class and taken in turn, so one large-area class
# (e.g. Weathering) cannot dominate the averages. Each image is used once.
rng = np.random.default_rng(0)
min_area = 0.002 * cfg.img_size ** 2
cands = {c: [] for c in TARGET_CLASSES}
for name in rng.permutation(SPLIT["test"]):
    m = load_mask(name, cfg.img_size)
    for c in TARGET_CLASSES:
        if m[CLASS_INDEX[c]].sum() >= min_area:
            cands[c].append(name)
xai_items, used = [], set()
while len(xai_items) < cfg.n_xai_images and any(cands.values()):
    for c in TARGET_CLASSES:
        while cands[c] and cands[c][0] in used:
            cands[c].pop(0)
        if cands[c] and len(xai_items) < cfg.n_xai_images:
            name = cands[c].pop(0); used.add(name); xai_items.append((name, CLASS_INDEX[c]))
print(f"{len(xai_items)} images selected:", pd.Series([CLASSES[c] for _, c in xai_items]).value_counts().to_dict())

rand_model = randomized_copy(xai_model)
xai_rows, gallery, gallery_fallback = [], [], []
for name, c in tqdm(xai_items, desc="Explanations"):
    x = torch.from_numpy(((load_image(name, cfg.img_size).astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
    gt = torch.from_numpy(load_mask(name, cfg.img_size)[c]).to(DEVICE).float()
    with torch.no_grad():
        prob = torch.sigmoid(xai_model(x)["logits"][0, c].float())
    M = (prob > xai_thr[c]).float(); region = "pred"
    if M.sum() < 20: M, region = gt, "gt"
    base = blurred(x)
    maps = {"gradcam": seg_gradcam(xai_model, x, c, M),
            "occlusion": occlusion_map(xai_model, x, c, M, base, cfg.occlusion_patch),
            "random": torch.rand(x.shape[-2:], device=DEVICE)}
    rand_maps = {"gradcam": seg_gradcam(rand_model, x, c, M),
                 "occlusion": occlusion_map(rand_model, x, c, M, base, cfg.occlusion_patch)}
    for meth, a in maps.items():
        d_auc, i_auc, sd, si = deletion_insertion(xai_model, x, base, a, c, M)
        pos = a.clamp_min(0)
        xai_rows.append({"image": name, "class": CLASSES[c], "region": region, "method": meth,
                         "deletion_AUC": d_auc, "insertion_AUC": i_auc,
                         "leakage": float(pos[M < 0.5].sum() / (pos.sum() + 1e-8)),
                         "rand_model_rank_corr": rank_corr(a, rand_maps[meth]) if meth in rand_maps else np.nan})
    # gallery: three examples of different classes in which the model actually predicts the target class
    item = (name, c, x, gt, prob, maps["gradcam"], maps["occlusion"])
    if region == "pred" and len(gallery) < 3 and all(g[1] != c for g in gallery):
        gallery.append(item)
    elif region == "gt" and len(gallery_fallback) < 3:
        gallery_fallback.append(item)

for item in gallery_fallback:                 # only if fewer than three predicted examples exist
    if len(gallery) < 3 and all(g[1] != item[1] for g in gallery):
        gallery.append(item)
xai = pd.DataFrame(xai_rows)
xai.to_csv(os.path.join(TAB_DIR, "e5_xai_per_image_raw.csv"), index=False)
xai_summary = xai.groupby("method")[["deletion_AUC", "insertion_AUC", "leakage", "rand_model_rank_corr"]].agg(["mean", "std"])
print(xai_summary.round(4).to_string())

**Reading the table.** A method is usable only if its deletion AUC is clearly *below* and its insertion AUC clearly *above* the `random` row, and its rank correlation with the randomised model is low (say below 0.3). If Seg-Grad-CAM fails the randomisation test, its maps reflect the architecture rather than what the model learned, and they should not appear in the paper as evidence of model reasoning.

In [ ]:
# CELL 33 · Table 8 and Table S4: faithfulness of explanations
from scipy.stats import wilcoxon
piv = xai.pivot_table(index="image", columns="method", values=["deletion_AUC", "insertion_AUC"])
pvals = {}
for meth in ["gradcam", "occlusion"]:
    for metric, alt in [("deletion_AUC", "less"), ("insertion_AUC", "greater")]:
        try:
            pvals[(meth, metric)] = float(wilcoxon(piv[(metric, meth)], piv[(metric, "random")], alternative=alt).pvalue)
        except ValueError:
            pvals[(meth, metric)] = np.nan

def _pm_col(meth, col, d=3):
    return pm(xai_summary.loc[meth, (col, "mean")], xai_summary.loc[meth, (col, "std")], d)

table8 = pd.DataFrame([{
    "Method": XAI_NAMES[m],
    "Deletion AUC (↓)": _pm_col(m, "deletion_AUC"),
    "Insertion AUC (↑)": _pm_col(m, "insertion_AUC"),
    "Off-target leakage": _pm_col(m, "leakage"),
    "Rank corr., random model (↓)": "–" if m == "random" else _pm_col(m, "rand_model_rank_corr"),
    "p, deletion": "–" if m == "random" else pfmt(pvals[(m, "deletion_AUC")]),
    "p, insertion": "–" if m == "random" else pfmt(pvals[(m, "insertion_AUC")]),
} for m in XAI_KEYS if m in xai_summary.index])
publish_table("xai", table8, raw=xai_summary, n_images=xai.image.nunique(), model=f"{MODEL_NAMES[BEST_MODEL]} (seed {E1_SEEDS[0]})")

leak = xai[xai.method != "random"].pivot_table(index="class", columns="method", values="leakage", aggfunc="mean")
table_s4 = pd.DataFrame({"Class": [CLASS_NAMES[c] for c in leak.index],
                         "Images": [str(int((xai[(xai.method == 'gradcam') & (xai['class'] == c)]).shape[0])) for c in leak.index]})
for m in ["gradcam", "occlusion"]:
    if m in leak.columns:
        table_s4[XAI_NAMES[m]] = [f3(v) for v in leak[m]]
publish_table("xai_leakage", table_s4, raw=leak, model=f"{MODEL_NAMES[BEST_MODEL]} (seed {E1_SEEDS[0]})")

In [ ]:
# CELL 34 · Figure 7: explanation gallery
import textwrap
VIEWS = ["Image and outline", "Probability", "Seg-Grad-CAM", "Occlusion"]

def _views(item):
    name, c, x, gt, prob, gc, occ = item
    img = denorm(x[0])
    edge = cv2.morphologyEx(gt.cpu().numpy().astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0
    outlined = img.copy(); outlined[edge] = (235, 104, 52)
    return img, outlined, prob.cpu().numpy(), gc, occ

def _heat(ax, img, a):
    a = a.clamp_min(0).cpu().numpy(); a = a / (a.max() + 1e-8)
    ax.imshow(a, cmap="magma", alpha=np.clip(a, 0, 1) * 0.85, vmin=0, vmax=1)

def _draw_view(ax, item, v, title=None, ylabel=None):
    img, outlined, prob, gc, occ = _views(item)
    if v == 0:
        image_axes(ax, outlined, title=title, ylabel=ylabel)
    elif v == 1:
        image_axes(ax, prob, title=title, ylabel=ylabel, cmap="Blues", vmin=0, vmax=1)
    else:
        image_axes(ax, img, title=title, ylabel=ylabel)
        _heat(ax, img, gc if v == 2 else occ)

def draw_gallery(layout):
    n = len(gallery)
    if layout == "horizontal":            # one example per row, four views per row
        cw = (COL2 - 0.3) / 4
        fig, axes = plt.subplots(n, 4, figsize=(COL2, n * cw + 0.3), squeeze=False)
        for i, item in enumerate(gallery):
            label = wrap_label(f"({'abc'[i]}) {CLASS_NAMES[CLASSES[item[1]]]}", 20)
            for v in range(4):
                _draw_view(axes[i, v], item, v, title=VIEWS[v] if i == 0 else None, ylabel=label if v == 0 else None)
    else:                                 # one example per column, four views per column
        cw = (COL1 - 0.3) / n
        fig, axes = plt.subplots(4, n, figsize=(COL1, 4 * cw + 0.55), squeeze=False)
        for j, item in enumerate(gallery):
            title = wrap_label(f"({'abc'[j]}) {CLASS_NAMES[CLASSES[item[1]]]}", 14)
            for v in range(4):
                _draw_view(axes[v, j], item, v, title=title if v == 0 else None, ylabel=VIEWS[v] if j == 0 else None)
    return fig

if gallery:
    publish_figure("xai_gallery", draw_gallery, model=f"{MODEL_NAMES[BEST_MODEL]} (seed {E1_SEEDS[0]})")

### 12.4 Which classes are confused with which?

Explanations show *why* a region is labelled as it is; the confusion matrix shows *where* the model goes wrong. Each row is a ground-truth class; each cell is the share of that class's pixels that the model **also or instead** predicts as the column class. Because labels are multi-label, rows need not sum to 1, and off-diagonal values can be legitimate (Rust on Exposed Rebars). The block for the confusable concrete defects is the one to discuss in the paper.

In [ ]:
# CELL 35 · Figure 8: class confusion
# Multi-label data: a pixel labelled both "Exposed rebars" and "Spalling" that is predicted as Spalling
# is correct, not a confusion. Off-diagonal entries therefore count only pixels that are NOT labelled
# with the predicted class; the diagonal is the per-class recall. One pass over the test set, cached.
conf_path = os.path.join(cfg.project_dir, "results", f"conf_exclusive_{BEST_MODEL}_s{E1_SEEDS[0]}.npy")
if os.path.exists(conf_path):
    conf = np.load(conf_path)
else:
    cm = load_trained(BEST_MODEL, E1_SEEDS[0])
    thr_c = torch.as_tensor(np.array(cfg.thr_grid)[E1[(BEST_MODEL, E1_SEEDS[0])]["thr"]], dtype=torch.float32).to(DEVICE).view(1, C, 1)
    num = torch.zeros(C, C, device=DEVICE, dtype=torch.float64); den = torch.zeros(C, device=DEVICE, dtype=torch.float64)
    for x, y, _ in tqdm(make_loader(SPLIT["test"], batch_size=max(1, cfg.batch_size // 2)), desc="confusion", leave=False):
        x, y = x.to(DEVICE), (y.to(DEVICE) > 0.5)
        p = predict([cm], x, "single")[0]
        B = x.shape[0]
        gt = y.reshape(B, C, -1).double(); pr = (p.reshape(B, C, -1) > thr_c).double()
        off = torch.einsum("bcn,bdn->cd", gt, pr * (1 - gt))           # predicted d where d is not labelled
        hit = (gt * pr).sum(dim=(0, 2))                                 # recall numerator
        num += off + torch.diag(hit)                                  # off-diagonal of `off` is the only non-zero part
        den += gt.sum(dim=(0, 2))
    conf = (num / den.clamp_min(1)[:, None]).cpu().numpy()
    np.save(conf_path, conf)
    del cm
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
focus = ["Spalling", "Rockpocket", "WConccor", "Cavity", "Efflorescence", "Weathering", "Wetspot",
         "Crack", "ACrack", "Rust", "ExposedRebars"]
fidx = [CLASS_INDEX[c] for c in focus]
sub = conf[np.ix_(fidx, fidx)]
pd.DataFrame(conf, index=CLASSES, columns=CLASSES).to_csv(os.path.join(TAB_DIR, "confusion_full_raw.csv"))

def draw_confusion(layout):
    big = layout == "horizontal"
    fig, ax = plt.subplots(1, 1, figsize=(COL15, 4.45) if big else (COL1, 4.6))
    im = ax.imshow(sub, cmap="Blues", vmin=0, vmax=1)
    names = [CLASS_NAMES[c] for c in focus]
    ax.set_xticks(range(len(focus))); ax.set_xticklabels(names, rotation=90, fontsize=7 if big else 6)
    ax.set_yticks(range(len(focus))); ax.set_yticklabels(names, fontsize=7 if big else 6)
    ax.xaxis.tick_top(); ax.xaxis.set_label_position("top")
    ax.tick_params(axis="both", length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    fs = 6 if big else 4.5
    for i in range(len(focus)):
        for j in range(len(focus)):
            ax.text(j, i, f"{sub[i, j]:.2f}", ha="center", va="center", fontsize=fs,
                    color="white" if sub[i, j] > 0.55 else INK)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("Ground-truth class")
    cb = fig.colorbar(im, ax=ax, location="right" if big else "bottom", fraction=0.046 if big else 0.05,
                      pad=0.02 if big else 0.03, shrink=0.9)
    cb.set_label("Share of ground-truth pixels", fontsize=7)
    cb.ax.tick_params(labelsize=6, width=0.5, length=2)
    cb.outline.set_linewidth(0.5)
    return fig

publish_figure("confusion", draw_confusion, model=f"{MODEL_NAMES[BEST_MODEL]} (seed {E1_SEEDS[0]})")

---
## 13. Robustness to image quality

Inspection photos are often blurred, noisy, under-exposed or heavily compressed. A trustworthy model should lose accuracy gracefully **and become more uncertain as quality drops**; a model that stays confident on a ruined image is dangerous.

We corrupt a fixed subset of test images at three severities and record mIoU (with the thresholds tuned on clean validation data) and mean uncertainty from MC dropout.

| Corruption | Severity 1 / 2 / 3 |
|---|---|
| Gaussian blur | σ = 1, 2, 4 px |
| Gaussian noise | σ = 8, 16, 32 grey levels |
| Low light | brightness × 0.7, 0.5, 0.3 |
| JPEG | quality 30, 15, 8 |

In [ ]:
# CELL 36 · Table 9 and Figure 9: robustness
rob_names = list(np.random.default_rng(1).permutation(SPLIT["test"])[: cfg.n_robust_images])
rob_model = load_trained(BEST_MODEL, E1_SEEDS[0])
rob_thr = E1[(BEST_MODEL, E1_SEEDS[0])]["thr"]
T_ROB = max(2, cfg.mc_samples // 2)
CORRUPTIONS = ["blur", "noise", "dark", "jpeg"]

rob_rows = []
ev = run_eval([rob_model], rob_names, "mc", f"rob_{BEST_MODEL}_clean", T=T_ROB)
clean = {"mIoU": miou(*select(ev, rob_thr)), "total_unc": ev["unc_img"][:, 0].mean(), "epistemic_unc": ev["unc_img"][:, 1].mean()}
for kind in CORRUPTIONS:
    rob_rows.append({"corruption": kind, "severity": 0, **clean})
    for sev in (1, 2, 3):
        ev = run_eval([rob_model], rob_names, "mc", f"rob_{BEST_MODEL}_{kind}{sev}", T=T_ROB, corruption=(kind, sev))
        rob_rows.append({"corruption": kind, "severity": sev, "mIoU": miou(*select(ev, rob_thr)),
                         "total_unc": ev["unc_img"][:, 0].mean(), "epistemic_unc": ev["unc_img"][:, 1].mean()})
rob = pd.DataFrame(rob_rows)
rows9 = [{"Corruption": "None (clean)", "Severity": "0", "mIoU": f3(clean["mIoU"]),
          "Total uncertainty": f3(clean["total_unc"], 4), "Epistemic uncertainty": f3(clean["epistemic_unc"], 5)}]
for r in rob[rob.severity > 0].itertuples():
    rows9.append({"Corruption": CORRUPTION_NAMES[r.corruption], "Severity": str(r.severity), "mIoU": f3(r.mIoU),
                  "Total uncertainty": f3(r.total_unc, 4), "Epistemic uncertainty": f3(r.epistemic_unc, 5)})
publish_table("robustness", pd.DataFrame(rows9), raw=rob, model=f"{MODEL_NAMES[BEST_MODEL]} (seed {E1_SEEDS[0]})", n_images=len(rob_names))

def draw_robustness(layout):
    if layout == "horizontal":
        fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.7))
    else:
        fig, axes = plt.subplots(2, 1, figsize=(COL1, 4.9))
    for kind in CORRUPTIONS:
        st = style_of("corruption", kind)
        r = rob[rob.corruption == kind].sort_values("severity")
        axes[0].plot(r.severity, r.mIoU, color=st["color"], marker=st["marker"], linestyle=st["linestyle"])
        axes[1].plot(r.severity, r.epistemic_unc * 1e4, color=st["color"], marker=st["marker"], linestyle=st["linestyle"])
    axes[0].set(title="(a) Accuracy", xlabel="Severity (0 = clean)", ylabel="mIoU", xticks=[0, 1, 2, 3])
    axes[1].set(title="(b) Epistemic uncertainty", xlabel="Severity (0 = clean)",
                ylabel=r"Mean epistemic uncertainty ($\times 10^{-4}$)",
                xticks=[0, 1, 2, 3])
    legend_below(fig, [series_handle("corruption", k, CORRUPTION_NAMES[k]) for k in CORRUPTIONS],
                 [CORRUPTION_NAMES[k] for k in CORRUPTIONS])
    return fig

publish_figure("robustness", draw_robustness, model=f"{MODEL_NAMES[BEST_MODEL]} (seed {E1_SEEDS[0]})")

---
## 14. Results summary for the paper

This section prints the headline numbers, lists every figure and table in paper order with its layout-check result, records the exact software versions, and writes the **Word report** (`dacl10k_figures_and_tables.docx`) with all tables and figures in numbered order.

**Before writing, check these points against the results:**

1. **RQ1:** Does any model beat the replicated baseline with a Holm-adjusted p < 0.05 *on both* `test` and `test_clean`? Report the replicated baseline next to the published 0.424 mIoU, and note that the published figure used dataset v1 (18 classes), the authors' own test split and possibly a different metric averaging, so the two are not directly comparable.
2. **Leakage:** How large is the gap between `test` and `test_clean`? A large gap is a finding about the benchmark, worth its own paragraph.
3. **RQ2:** Which explanation method passes all three tests (Table 8)? Use only those maps in figures, and describe them as model diagnostics, not engineering justification.
4. **RQ3:** Which method has the lowest ECE *after* temperature scaling (Table 6), and how far from 1 are the fitted temperatures (Table S2)?
5. **RQ4:** How much more than random review does the best method find at a 10% review budget (Table 7)?
6. **Robustness:** Does uncertainty rise as accuracy falls for every corruption type?

In [ ]:
# CELL 37 · Headline results, Word report and file index
best_cal = calib[calib.stage == "temp-scaled"].sort_values("ECE_active").method.iloc[0]   # ECE (active) is the reported metric
insp = e4[e4.view == "inspector (any class)"].set_index("method")
headline = {
    "Best model (selected on validation mIoU)": MODEL_NAMES[BEST_MODEL],
    "Seeds per model": N_SEEDS,
    "Best model, test mIoU": round(float(summary.loc[BEST_MODEL, "test_mIoU mean"]), 4),
    "Best model, leakage-free test mIoU": round(float(summary.loc[BEST_MODEL, "test_clean_mIoU mean"]), 4),
    "Replicated baseline, test mIoU": round(float(summary.loc["fpn_b4", "test_mIoU mean"]), 4) if "fpn_b4" in summary.index else None,
    "Published baseline, test mIoU (Flotzinger et al., 2024; dataset v1)": 0.424,
    "Near-duplicate pairs across official splits": int(len(cross)),
    "Best-calibrated method after temperature scaling (ECE, active pixels)": METHOD_NAMES[best_cal],
    "Share of errors found reviewing 10% of pixels (highest of the methods)": round(float(insp["recall@10%"].max()), 4),
}
for k, v in headline.items():
    print(f"{k:72s} {v}")
json.dump(headline, open(os.path.join(cfg.project_dir, "results", "headline.json"), "w"), indent=2)

versions = {"python": sys.version.split()[0], "torch": torch.__version__, "smp": smp.__version__,
            "transformers": transformers.__version__, "peft": peft.__version__, "albumentations": A.__version__,
            "numpy": np.__version__, "matplotlib": mpl.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "smoke_test": SMOKE_TEST}
json.dump(versions, open(os.path.join(cfg.project_dir, "results", "environment.json"), "w"), indent=2)

# ---- every figure and table in numbered order ----
print("\nPublished items (in paper order):")
items = sorted(list(PUBLISHED["tables"].values()) + list(PUBLISHED["figures"].values()), key=lambda v: _order_key(v["label"]))
n_warn = 0
for v in items:
    issues = sum(len(x) for x in v.get("issues", {}).values())
    n_warn += issues
    print(f"  {v['label']:10s} {'layout check passed' if 'issues' in v and not issues else ('table' if 'issues' not in v else f'{issues} layout warning(s)')}")
expected = [l for l, _ in FIGURES.values()] + [l for l, _ in TABLES.values()]
missing = [l for l in expected if l not in {v["label"] for v in items}]
if missing:
    print("Not produced (optional or skipped):", ", ".join(missing))
print(f"Layout warnings in total: {n_warn}")

index_path = write_index(os.path.join(cfg.project_dir, "figure_and_table_list.md"))
try:
    report = build_word_report(os.path.join(cfg.project_dir, "dacl10k_figures_and_tables.docx"))
    print("\nWord report:", report)
except ImportError:
    print("\npython-docx is not installed; run CELL 01 again to create the Word report.")
print("Figure and table list:", index_path)
print("Figures folder:", FIG_DIR)
print("Tables folder :", TAB_DIR)

---
## 15. Limitations of this implementation and next steps

- **Grouping by near-duplicates is a proxy.** Without bridge identifiers we cannot guarantee that two dissimilar photos of the same bridge are in one split. Ask the dataset authors whether source identifiers exist; if they do, replace the pHash groups with them.
- **Coarse crack labels.** dacl10k crack polygons are wider than the cracks. IoU therefore penalises thin, correct predictions, and calibration targets for Crack are noisy. Consider reporting a boundary-tolerant metric (for example, IoU after dilating both masks by 2 px) for Crack.
- **Pixel sampling for calibration.** Calibration metrics use a uniform pixel sample per image, dominated by background. The `ECE_active` column addresses this partly; for a deeper analysis, add class-balanced sampling.
- **Ensemble size.** The ensemble reuses the three E1 seeds. A larger ensemble can be built by training more seeds of the best model (add them to `cfg.seeds` and re-run 9.2 and 10).
- **Official test servers.** For a leaderboard comparison, predictions on `testdev`/`testchallenge` can be submitted to the organisers' evaluation server listed in the dacl10k-toolkit README; check whether it still accepts submissions.
- **Licence.** CC BY-NC 4.0: trained weights and results are for non-commercial research only.

### References

The complete, verified reference list is in the paper. Main methods used in this notebook:

- J. Flotzinger, P.J. Rösch, T. Braml, dacl10k: Benchmark for semantic bridge damage segmentation, in: Proceedings of the IEEE/CVF Winter Conference on Applications of Computer Vision (WACV), 2024.
- M. Oquab, T. Darcet, T. Moutakanni, H.V. Vo, M. Szafraniec, V. Khalidov, P. Fernandez, D. Haziza, F. Massa, A. El-Nouby, M. Assran, N. Ballas, W. Galuba, R. Howes, P.-Y. Huang, S.-W. Li, I. Misra, M. Rabbat, V. Sharma, G. Synnaeve, H. Xu, H. Jegou, J. Mairal, P. Labatut, A. Joulin, P. Bojanowski, DINOv2: Learning robust visual features without supervision, Transactions on Machine Learning Research (2024).
- E.J. Hu, Y. Shen, P. Wallis, Z. Allen-Zhu, Y. Li, S. Wang, L. Wang, W. Chen, LoRA: Low-rank adaptation of large language models, in: International Conference on Learning Representations (ICLR), 2022.
- E. Xie, W. Wang, Z. Yu, A. Anandkumar, J.M. Alvarez, P. Luo, SegFormer: Simple and efficient design for semantic segmentation with transformers, in: Advances in Neural Information Processing Systems 34 (NeurIPS 2021), 2021.
- C. Guo, G. Pleiss, Y. Sun, K.Q. Weinberger, On calibration of modern neural networks, in: Proceedings of the 34th International Conference on Machine Learning (ICML), 2017.
- B. Lakshminarayanan, A. Pritzel, C. Blundell, Simple and scalable predictive uncertainty estimation using deep ensembles, in: Advances in Neural Information Processing Systems 30 (NIPS 2017), 2017.
- Y. Gal, Z. Ghahramani, Dropout as a Bayesian approximation: Representing model uncertainty in deep learning, in: Proceedings of the 33rd International Conference on Machine Learning (ICML), 2016.
- K. Vinogradova, A. Dibrov, G. Myers, Towards interpretable semantic segmentation via gradient-weighted class activation mapping (student abstract), in: Proceedings of the AAAI Conference on Artificial Intelligence, 2020.
- J. Adebayo, J. Gilmer, M. Muelly, I. Goodfellow, M. Hardt, B. Kim, Sanity checks for saliency maps, in: Advances in Neural Information Processing Systems 31 (NeurIPS 2018), 2018.
- V. Petsiuk, A. Das, K. Saenko, RISE: Randomized input sampling for explanation of black-box models, in: British Machine Vision Conference (BMVC), 2018.


---
## Release the GPU

The last cell disconnects this Colab runtime once every cell above has finished, so no compute units are used while nobody is watching. Everything is already saved on Google Drive. Set `RELEASE_GPU_AT_END = False` if you want to keep working in this session; after a dry run (`SMOKE_TEST = True`) the GPU is always kept. If an earlier cell stops with an error, this cell does not run, and Colab ends the idle session on its own later.

In [ ]:
# CELL 38 · Release the GPU when everything has finished
RELEASE_GPU_AT_END = True    # <<< False keeps the session running

if IN_COLAB and RELEASE_GPU_AT_END and not SMOKE_TEST:   # after a dry run the session is kept for the real run
    print("All cells finished; results are on Drive. Releasing the GPU now.")
    from google.colab import runtime
    runtime.unassign()
else:
    print("GPU kept (dry run, not in Colab, or RELEASE_GPU_AT_END = False).")